# Reproducing Strickland and McDannald (2022) from CRCNS bst-3

**Paper.** Strickland JA, McDannald MA (2022). Brainstem networks construct threat probability and
prediction error from neuronal building blocks. *Nature Communications* 13:6192.
[doi:10.1038/s41467-022-34021-1](https://doi.org/10.1038/s41467-022-34021-1). The claims file
(`claims/strickland-2022.yaml`) was written from the version of record (PMC9582012) with its
Supplementary Information and Source Data workbook.

**Data set.** CRCNS [bst-3](https://crcns.org/data-sets/brainstem/bst-3),
[doi:10.6080/K0Q23XGR](https://doi.org/10.6080/K0Q23XGR), "Neuropixels recordings from 21 brainstem
regions in rats during fear discrimination", read from the public bucket
(`crcnsarchive/bst-3/live/data.zip`, 367,964,637 bytes, last modified 2026-02-11). CRCNS data sets
are not versioned, so the version is recorded as `unversioned`; the MD5 sums of the bucket's
`checksums.md5` are printed below (data.zip `4ee6c433f5255693d5e85bcca6819cd4`), so that a later
change can be detected. The paper cites the data set by its DOI and no version; there is only the
one published copy (`live/`, descriptor version 0.6 of 2022-11-29), so it is the one pinned.

**What the data set holds.** One MATLAB file per single unit (1812, spike times and the session's
cue, shock, poke and pellet times, in samples at 30 kHz), the cube file the paper's `cubeScript.m`
builds from them (binned and normalized firing per trial, suppression ratios, region and
coordinates per unit), and the paper's five analysis scripts. It holds no cluster membership.

**Outside input.** The paper's own cue and outcome clusters (L1) come from its Source Data workbook
(supplementary file 41467_2022_34021_MOESM4_ESM.xlsx of doi:10.1038/s41467-022-34021-1, streamed from
the publisher, MD5 checked), matched to the units by their firing rows (claim
`source-data-cluster-link`). It names clusters for units the data set already holds; it brings in no
measurement. The analyses run on the data set's data.

**The paper's code.** The scripts in the data set (`data/scripts`) are the code the paper's Code
availability points to. Where the Methods leave a detail open and a script settles it, the script is
followed and cited (see the docstring of `bst3.py`).

| Claim | Tested |
|---|---|
| source-data-cluster-link | tested (link) |
| fig1c-suppression-order | tested |
| figs2-each-rat-discriminates | tested |
| units-sessions-subjects | tested |
| fig1e-regions | tested |
| fig2b-cue-clusters | tested |
| fig2c-cue-pc1-ordered | tested |
| fig2c-pc1-contribution | tested |
| fig2d-danger-latency | tested |
| fig2e-cue-subnetwork | tested |
| fig2f-cue-hubs | tested |
| fig2g-cue-anatomy | tested |
| fig3a-cluster-signals | tested, with the Source Data labels |
| fig3b-signal-pca | tested |
| fig3c-subnetwork-lesion | tested |
| fig3d-supranetwork-lesion | tested |
| fig3e-network-anatomy | tested |
| fig4b-outcome-clusters | tested |
| fig4c-positive-prediction-error | tested |
| fig4b-phasic-network | tested, with the Source Data labels |
| fig4b-tonic-network | tested, with the Source Data labels |
| fig4e-network-pca | tested |
| fig4f-outcome-hubs | tested, with the Source Data labels |
| fig4g-outcome-anatomy | tested |
| fig5a-outcome-cluster-signals | tested, with the Source Data labels |
| fig5b-outcome-signal-pca | tested |
| fig5c-phasic-lesion | tested |
| fig5d-tonic-lesion | tested |
| fig5e-no-anatomical-bias | tested |
| cue-outcome-membership-independent | tested |

No claim is blocked. Random seeds are fixed per section.

In [ ]:
import time
import resource
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import linear_sum_assignment

import bst3

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)
started = time.time()

print(f"Data set CRCNS {bst3.DATASET}, version {bst3.VERSION}, doi:{bst3.DOI}")
print("checksums.md5 in the bucket:")
print(bst3.read_text("checksums.md5"))
zf = bst3.open_zip()
members = zf.namelist()
print(len(members), "members in data.zip;", Counter(m.split("/")[1] if m.count("/") > 1 else m for m in members))
for m in members:
    if not m.startswith("data/singleUnits/") and not m.endswith("/"):
        print("  ", m, zf.getinfo(m).file_size, "bytes")
units = bst3.scan_units(zf)
cube = bst3.load_cube(zf)
sheets, sd_md5 = bst3.load_source_data()
print(f"Source Data workbook md5 {sd_md5} (expected {bst3.SOURCE_DATA_MD5}): {sd_md5 == bst3.SOURCE_DATA_MD5}")
print(f"loaded in {time.time() - started:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

What a single-unit file holds, what the cube holds, and checks of the description against the values.

In [ ]:
# One single-unit file of each format, listed in full
for fmt in ("MATLAB 5.0", "MATLAB 7.3"):
    name = next(n for n, f in zip(units["ID"], units["format"]) if f == fmt)
    uid, spike, ev = bst3._parse_unit(zf.read(f"data/singleUnits/{name}.mat"))
    print(f"{fmt} file {name}.mat: id {uid!r}, spike {spike.shape} {spike.dtype} "
          f"(first {spike[:3].tolist()}, last {spike[-1]}), events:")
    for k, v in ev.items():
        print(f"    time.{k}: {v.size} values, {v.min() if v.size else None} to {v.max() if v.size else None}")
print("\nFiles by format:", dict(Counter(units["format"])))
print("\nCube fields read (shape as unit x bin x trial):")
for k in bst3.CUBE_FIELDS:
    if k != "anatomy/ID":
        print(f"    {k}: {cube[k].shape}")
print("The cube's anatomy.region is a MATLAB string object (MCOS), not readable outside MATLAB; region names come")
print("from anatomy.regionNum and the numbering in cubeScript.m.")

In [ ]:
# Units, sessions, subjects
ids = cube["ID"]
order = {u: i for i, u in enumerate(units["ID"])}
assert len(order) == len(ids) == len(set(ids))
ix = np.array([order[u] for u in ids])           # single-unit scan -> cube order
rat = np.array([u[2:4] for u in ids])
session = np.array([u[:10] for u in ids])
subject = np.array([bst3.RATS.index(r) + 1 for r in rat])
female = np.isin(rat, list(bst3.FEMALE))
print("Unit files:", len(units["ID"]), "; cube units:", len(ids), "; same set of ids:", set(ids) == set(units["ID"]))
print("Distinct sessions (animal + date in the id):", len(set(session)))
tab = pd.DataFrame({"subject": subject, "animal": rat, "female": female, "session": session})
per = tab.groupby("subject").agg(animal=("animal", "first"), female=("female", "first"),
                                 units=("session", "size"), sessions=("session", "nunique"))
per["sessions, Supp. Table 1"] = [8, 9, 10, 2, 8, 1, 10, 7, 10, 10]
print(per.to_string())
print("Units from females:", int(female.sum()))

# Duplicate recordings: identical spike-count arrays (250-ms bins over all trials) between units
h = Counter(units["c250"][i].tobytes() for i in range(len(ix)))
print("\nUnit pairs with identical binned spike counts:", sum(v * (v - 1) // 2 for v in h.values() if v > 1))
print("Units with repeated spike times:", int((units["n_duplicate_spikes"] > 0).sum()),
      "; repeated times in total:", int(units["n_duplicate_spikes"].sum()),
      "; files with unsorted spike times:", int(units["unsorted"].sum()))

In [ ]:
# The cube against the spike times, and the trial structure
c1s = np.transpose(units["c1s"][ix], (0, 2, 1))
print("cube fire.raw.s1 equals spike counts in the script's 1-s bins (each spike time counted once):",
      bool(np.array_equal(c1s, cube["fire/raw/s1/interval"])))
print("cube fire.raw.base2 equals floor(count in 2 s before onset / 2):",
      bool(np.array_equal(units["base2"][ix], cube["fire/raw/base2"][:, 0, :])))
diff = cube["fire/raw/s1/interval"] - cube["fire/raw/base2"][:, 0, :][:, None, :]
zs = (diff - diff.mean((1, 2), keepdims=True)) / diff.std((1, 2), ddof=1, keepdims=True)
print("cube fire.z.s1 equals the script's z of differential firing, max abs difference:",
      float(np.abs(zs - cube["fire/z/s1/interval"]).max()))
S = units["sessions"]
ok = 0
for k, s in S.items():
    sr = bst3.suppression_ratios(s["cue"], s["poke"], "code")
    ok += np.allclose(cube["poke/SR"][session == k, 0, :], sr[None, :])
print(f"cube poke.SR equals the ratios recomputed from poke times in {ok} of {len(S)} sessions")
for k, s in S.items():
    if "differing_units" in s:
        print(f"Session {k} ({int((session == k).sum())} units): units {s['differing_units']} carry cue or poke times "
              f"that differ from unit {s['file_id']} (the other units agree with it)")
        for u in s["differing_units"] + [s["file_id"]]:
            _, _, ev = bst3._parse_unit(zf.read(f"data/singleUnits/{u}.mat"))
            cue = np.concatenate([ev["cue100"], ev["cueSPR"], ev["cueNsPR"], ev["cue0"]])
            o = np.argsort(s["cue"])
            print(f"    {u}: pokes {ev['poke'].size}; reference minus this unit's cue onsets, in session order (s): "
                  f"{np.round((s['cue'] - cue)[o] / bst3.FS, 3).tolist()}")

rows = []
for k, s in S.items():
    cue = s["cue"]
    nearest = [int(np.argmin(np.abs(sh - cue))) for sh in s["shock"]]
    rows.append({"session": k, "trials": len(cue), "shocks": len(s["shock"]),
                 "shock rows": "".join(str(r + 1) + "," for r in sorted(nearest)),
                 "shock delay s": sorted(set(np.round((s["shock"] - cue[nearest]) / bst3.FS, 3))),
                 "session min": (cue.max() - cue.min()) / bst3.FS / 60,
                 "ITI min": np.diff(np.sort(cue)).mean() / bst3.FS / 60, "pokes": len(s["poke"])})
trials = pd.DataFrame(rows)
print("\nTrials per session:", dict(Counter(trials.trials)), "; shocks per session:", dict(Counter(trials.shocks)))
print("Rows of the trial table that have a shock:", dict(Counter(trials["shock rows"])))
print("Delay from cue onset to shock onset (s):", sorted({d for x in trials["shock delay s"] for d in x}))
print("Mean ITI per session, min: %.2f to %.2f" % (trials["ITI min"].min(), trials["ITI min"].max()))
print(trials[trials.shocks != 6].to_string(index=False))

# fire.z.ms500, the source of every outcome result, against the spike times: the script's 500-ms
# rate (count x 2) over the 40 s from 10 s before onset, minus base2, z-scored over trials and bins
c500 = units["c250"][ix][:, :, 40:200].reshape(len(ix), 16, 80, 2).sum(3) * 2
d500 = np.transpose(c500, (0, 2, 1)) - units["base2"][ix][:, None, :]
z500 = (d500 - d500.mean((1, 2), keepdims=True)) / d500.std((1, 2), ddof=1, keepdims=True)
print("cube fire.z.ms500 equals the script's z of differential 500-ms firing, max abs difference:",
      float(np.abs(z500 - cube["fire/z/ms500/interval"]).max()))

The trial rows are as the descriptor says: the shocks fall on rows 1 to 6 (danger, then
uncertainty shock), 12.0 s after cue onset (2 s after the 10-s cue), so shock offset is at
12.5 s, which is where the outcome window of `outcomeAnalysis.m` (500-ms bins 46 to 65 of the
40-s window from 10 s before cue onset) starts. Spike times and event times are in samples at
30 kHz, as `cubeScript.m` counts them.

In [ ]:
# Clock check: firing in the 2 s after shock onset minus the 2 s before cue onset, on shocked and
# unshocked rows, at the true event times and with every trial's times shifted by +60 s
# from the 250-ms counts (20 s before to 40 s after onset): shock onset is bins 128-135 (12-14 s)
c = units["c250"][ix].astype(float)              # unit, trial, bin
resp = c[:, :, 128:136].sum(2) / 2 - c[:, :, 72:80].sum(2) / 2   # Hz change
shock_rows, noshock_rows = resp[:, :6].mean(1), resp[:, 6:].mean(1)
d = shock_rows - noshock_rows
print(f"Per unit, firing change 12-14 s after onset, shock rows minus no-shock rows: mean {d.mean():.2f} Hz, "
      f"median {np.median(d):.2f} Hz, Wilcoxon p = {stats.wilcoxon(d).pvalue:.2g}")
# the same contrast 2 s earlier (10-12 s, the delay before shock) and in a window 20 s before onset
for name, sl in (("10-12 s (before shock)", slice(120, 128)), ("-20 to -18 s", slice(0, 8))):
    r = c[:, :, sl].sum(2) / 2 - c[:, :, 72:80].sum(2) / 2
    dd = r[:, :6].mean(1) - r[:, 6:].mean(1)
    print(f"Same contrast at {name}: mean {dd.mean():.2f} Hz, Wilcoxon p = {stats.wilcoxon(dd).pvalue:.2g}")

The shock response appears only after shock onset and only on the rows the trial table marks as
shocked, so the spike times, the event times and the trial rows share one clock and one order in
every file read here (a contrast between conditions, protocol step 2). This is shown over all units
together; per session it is consistent with the data, not shown.

In [ ]:
# Regions, coordinates, ranges of the variables used
reg_raw = cube["anatomy/regionNum"][:, 0, 0].astype(int)
region = np.array([bst3.REGION_REMAP[r] for r in reg_raw])
print("Raw region numbers (cubeScript.m numbering):", dict(sorted(Counter(reg_raw).items())))
print("After the scripts' merges (21 -> DpWh, 24 -> x/scp):",
      {bst3.REGIONS[k - 1]: v for k, v in sorted(Counter(region).items())})
for k in ("anatomy/x", "anatomy/y", "anatomy/z"):
    v = cube[k].ravel()
    print(f"{k}: {v.min():.2f} to {v.max():.2f} (units not stated)")
zs1, z5 = cube["fire/z/s1/interval"], cube["fire/z/ms500/interval"]
sr = cube["poke/SR"][:, 0, :]
print(f"z.s1: {zs1.shape}, {np.nanmin(zs1):.2f} to {np.nanmax(zs1):.2f}, NaN {int(np.isnan(zs1).sum())}")
print(f"z.ms500: {z5.shape}, {np.nanmin(z5):.2f} to {np.nanmax(z5):.2f}, NaN {int(np.isnan(z5).sum())}")
print(f"poke.SR: {sr.min():.2f} to {sr.max():.2f}; poke.base {cube['poke/base'].min():.0f} to {cube['poke/base'].max():.0f} "
      f"pokes/min; poke.cue {cube['poke/cue'].min():.0f} to {cube['poke/cue'].max():.0f} pokes/min")
X_cue = bst3.cue_matrix(zs1)
X_out = bst3.outcome_matrix(z5)
print("Cue matrix", X_cue.shape, "; outcome matrix", X_out.shape)

The passages of the paper's scripts that the analysis follows, printed from the data set, so that
each choice cited in `bst3.py` can be checked against its source. The `Shuffle` function the scripts
call is not among them, and none of them computes the danger latency of Fig. 2d.

In [ ]:
# The passages of the paper's scripts (data/scripts in data.zip) that the analysis follows, printed
# from the data set so that each choice cited in bst3.py can be checked
def show(name, first, last):
    lines = zf.read(f"data/scripts/{name}").decode("latin-1").splitlines()
    print(f"--- {name}, lines {first}-{last}")
    for n in range(first, last + 1):
        print(f"{n:5d}  {lines[n - 1]}")
show("cubeScript.m", 440, 449)        # 500-ms bins: count x 2, 40 s from 10 s before onset
show("cubeScript.m", 414, 420)        # suppression ratio, a trial with no pokes set to 1
show("cubeScript.m", 485, 500)        # differential firing (minus base2), z over all trials and bins
show("cueAnalysis.m", 20, 48)         # region renumbering, CIC (21) to DpWh, 11 and 24 merged
show("cueAnalysis.m", 1231, 1256)     # animal js10 ... js27 to subject 1 ... 10
show("cueAnalysis.m", 633, 640)       # PC1 contribution: each cluster's rows Shuffle(..., 2), 1000 times
show("cueAnalysis.m", 1104, 1116)     # cue regression: intercept, threat probability, mean suppression ratio
show("cueRegression.m", 211, 234)     # cue lesion: k1-k9 mean firing Shuffle(regData, 2) before regression
show("outcomeRegression.m", 43, 62)   # outcome regression: prediction error betas in columns 1-20, shock in 21-40
show("outcomeRegression.m", 106, 113) # outcome lesion: k1-k3 mean firing Shuffle(regData, 1)
show("outcomeAnalysis.m", 760, 770)   # cluster-cluster correlations, plotted squared
show("outcomeAnalysis.m", 800, 804)   # outcome hub measure: r and r squared of a unit with each fellow cluster
show("outcomeAnalysis.m", 934, 936)   # Fig. 4f plot labels hard-coded as 142, 171, 111, 189 units
scripts = {m: zf.read(m).decode("latin-1") for m in zf.namelist() if m.startswith("data/scripts/")}
for word in ("findchangepts", "latency", "function "):
    print(f"scripts containing '{word}':", [m.split("/")[-1] for m, s in scripts.items() if word in s.lower()])

## source-data-cluster-link

> The data set's units can be matched to the rows of the paper's Source Data cluster sheets, which
> gives the paper's own cue and outcome cluster membership.

Each unit's 42-value cue row and 80-value outcome row are compared with every row of Source Data
sheets "Figure 2b k1" to "k21" and "Figure 4b k1" to "k11". The rows are computed under the claims
file's four normalization readings (from the spike times, 250-ms bins) and under the normalization
of the paper's own `cubeScript.m`, which is what the cube holds (1-s and 500-ms bins z-scored
directly over the 40-s window from 10 s before onset, baseline rounded down).

In [ ]:
sd_cue = {k: bst3.sd_cue_cluster_rows(sheets, k) for k in range(1, 22)}
sd_out = {k: bst3.sd_outcome_cluster_rows(sheets, k) for k in range(1, 12)}
print("Source Data rows: cue", sum(len(v) for v in sd_cue.values()), "; outcome", sum(len(v) for v in sd_out.values()))
readings = {"cubeScript.m (the cube)": (X_cue, X_out)}
readings.update(bst3.normalized_readings(units["c250"][ix].transpose(0, 1, 2)))
link_rows = []
L1_cue = np.zeros(len(ids), int)
L1_out = np.zeros(len(ids), int)
for name, (rc, ro) in readings.items():
    res = {}
    for which, sd, rows in (("cue", sd_cue, rc), ("outcome", sd_out, ro)):
        exact, corr, lab = 0, 0, np.zeros(len(ids), int)
        n = 0
        for k, B in sd.items():
            hits, br, bi = bst3.match_rows(B, rows)
            n += len(B)
            for h, r, i in zip(hits, br, bi):
                if len(h) == 1:
                    exact += 1
                    lab[h[0]] = k
                if r > 0.99:
                    corr += 1
        res[which] = (exact, corr, n, lab)
    link_rows.append({"reading": name,
                      "cue rows matching one unit (max abs diff < 0.01)": f"{res['cue'][0]} / {res['cue'][2]}",
                      "cue rows with best r > 0.99": res["cue"][1],
                      "outcome rows matching one unit": f"{res['outcome'][0]} / {res['outcome'][2]}",
                      "outcome rows with best r > 0.99": res["outcome"][1]})
    if name.startswith("cubeScript"):
        L1_cue, L1_out = res["cue"][3], res["outcome"][3]
print(pd.DataFrame(link_rows).to_string(index=False))
print("\nUnits given a cue cluster:", int((L1_cue > 0).sum()), "; an outcome cluster:", int((L1_out > 0).sum()))
print("Cue cluster sizes (L1):", np.bincount(L1_cue, minlength=22)[1:].tolist())
print("Outcome cluster sizes (L1):", np.bincount(L1_out, minlength=12)[1:].tolist())
diffs = []
for k, B in sd_cue.items():
    hits, _, _ = bst3.match_rows(B, X_cue)
    diffs += [np.abs(X_cue[h[0]] - b).max() for h, b in zip(hits, B) if len(h) == 1]
print(f"Largest difference between a matched cue row and its Source Data row: {max(diffs):.2e}")

**Result: reproduced.** Under the normalization of the paper's own `cubeScript.m`, every one of the
1812 cue rows and 1812 outcome rows of the Source Data cluster sheets matches exactly one unit of
the data set, with differences at rounding level; the cluster sizes are the ones the claims file
lists. Under the claims file's four readings (n-a, n-b, w-a, w-b, from 250-ms bins) at most 2 rows
match within 0.01, though most correlate above 0.99 with a unit; the script's normalization is a detail the
Methods leave open and the paper's code settles. The link gives L1 for every claim below.

## fig1c-suppression-order

> During recording sessions suppression of rewarded nose poking was high to danger, intermediate
> to uncertainty and low to safety, with a main effect of cue (F(2,142) = 149.2, p = 1.26 x 10^-35)
> over 73 sessions.

Per session, the ratio per trial from its poke times; the session value per cue is the mean over
its trials (uncertainty pooling 8). Empty trials (no pokes before or during the cue) are set to 1
as in `cubeScript.m`, and to 0 (sr-a) or left out (sr-b) as the claims file's readings.

In [ ]:
keys = sorted(S)
sess_sex = np.array([k[2:4] in bst3.FEMALE for k in keys]).astype(int)
sd1c = np.array([r[1:4] for r in sheets["Figure 1c"][1:] if r[0] is not None], float)
print("Source Data Fig. 1c:", sd1c.shape[0], "sessions; means", sd1c.mean(0).round(3))
rows = []
vals = {}
for empty in ("code", "zero", "drop"):
    Y = []
    n_empty = 0
    for k in keys:
        r = bst3.suppression_ratios(S[k]["cue"], S[k]["poke"], empty)
        n_empty += int(np.isnan(bst3.suppression_ratios(S[k]["cue"], S[k]["poke"], "drop")).sum())
        Y.append([np.nanmean(r[0:4]), np.nanmean(r[4:12]), np.nanmean(r[12:16])])
    Y = np.array(Y)
    vals[empty] = Y
    a = bst3.mixed_anova_cue_by_group(Y, sess_sex)
    rows.append({"empty trials": {"code": "set to 1 (script)", "zero": "set to 0 (sr-a)", "drop": "left out (sr-b)"}[empty],
                 "danger": Y[:, 0].mean(), "uncertainty": Y[:, 1].mean(), "safety": Y[:, 2].mean(),
                 "F cue x sex (a-a)": a["F_mixed"], "df a-a": a["df_mixed"], "p a-a": a["p_mixed"],
                 "F repeated (a-b)": a["F_rm"], "df a-b": a["df_rm"], "p a-b": a["p_rm"]})
print("Trials with no pokes in either period:", n_empty, "of", 16 * len(keys))
print(pd.DataFrame(rows).round(4).to_string(index=False))
a = bst3.mixed_anova_cue_by_group(sd1c, np.zeros(len(sd1c)))
print(f"\nOn Source Data Fig. 1c: repeated-measures F({a['df_rm'][0]},{a['df_rm'][1]}) = {a['F_rm']:.1f}")
# do the data set's session values match the Source Data rows (as multisets)?
for empty, Y in vals.items():
    m = sum(np.any(np.all(np.abs(sd1c - y) < 1e-6, 1)) for y in Y)
    print(f"sessions whose three values equal a Source Data row ({empty}): {m} of {len(Y)}")
print("\nPer subject, mean over sessions (script reading):")
per_rat = pd.DataFrame(vals["code"], columns=["danger", "uncertainty", "safety"])
per_rat["subject"] = [bst3.RATS.index(k[2:4]) + 1 for k in keys]
print(per_rat.groupby("subject").agg(["mean", "size"]).round(3).to_string())

**Result: reproduced.** Over the 73 sessions the means are 0.914 (danger), 0.709 (uncertainty) and
0.230 (safety) with empty trials set to 1 as the script does, each equal to the Source Data mean, and
every session's three values equal a Source Data row. The cue x sex mixed ANOVA (a-a) gives
F(2,142) = 147.9 (149.2 +- 14.92), p < 1e-30. The other two treatments of empty trials give F 145.6
and 150.4 and the same order. The one-way repeated-measures reading (a-b) gives F(2,144) = 116.2, as
on the paper's own values, so the reported F fits the cue x sex design.

## figs2-each-rat-discriminates

> Rats showed complete discrimination during the recording sessions, shown rat by rat with 95%
> bootstrap confidence intervals over sessions.

Per rat, 10,000 bootstrap resamples of its sessions (percentile interval), on the script's ratios.

In [ ]:
rng = np.random.default_rng(2)
Y = vals["code"]
subj = np.array([bst3.RATS.index(k[2:4]) + 1 for k in keys])
rows = []
for s in range(1, 11):
    y = Y[subj == s]
    n = len(y)
    q = {"D-S": y[:, 0] - y[:, 2], "U-S": y[:, 1] - y[:, 2], "D-U": y[:, 0] - y[:, 1],
         "D": y[:, 0], "U": y[:, 1]}
    row = {"subject": s, "sessions": n, "danger": y[:, 0].mean(), "uncertainty": y[:, 1].mean(), "safety": y[:, 2].mean()}
    if n >= 2:
        b = rng.integers(0, n, (10000, n))
        for name, v in q.items():
            lo, hi = np.percentile(v[b].mean(1), [2.5, 97.5])
            row[name + " CI"] = f"[{lo:.2f}, {hi:.2f}]"
            row[name + " excl0"] = bool(lo > 0 or hi < 0)
    rows.append(row)
t = pd.DataFrame(rows)
print(t.round(3).to_string(index=False))
ok3 = t[t.sessions >= 3]
for name, cols in (("c-a (D-S and U-S exclude 0)", ["D-S excl0", "U-S excl0"]),
                   ("c-b (D-U and U-S exclude 0)", ["D-U excl0", "U-S excl0"]),
                   ("c-c (D and U exclude 0)", ["D excl0", "U excl0"])):
    n = int(ok3[cols].all(1).sum())
    print(f"{name}: {n} of {len(ok3)} rats with at least 3 sessions")

**Result: reproduced.** Under reading c-c (each rat's danger and uncertainty suppression intervals
exclude zero) all 8 rats with at least 3 sessions meet it. Under c-a 7 of 8 do (subject 9's
uncertainty minus safety interval, [-0.06, 0.28], includes zero) and under c-b 5 of 8. Subjects 4 and
6 (2 sessions and 1) have means ordered danger > uncertainty > safety (1.000, 0.887, 0.500 and 0.875,
0.521, 0.508).

## units-sessions-subjects

> 1812 single units were isolated and held over 73 one-hour sessions from ten rats, 965 of them from
> the four females, with per-subject totals of 192, 328, 141, 32, 52, 43, 116, 195, 409 and 304 units.

In [ ]:
expected = [192, 328, 141, 32, 52, 43, 116, 195, 409, 304]
got = [int((subject == s).sum()) for s in range(1, 11)]
print("units", len(ids), "(expected 1812); from females", int(female.sum()), "(expected 965)")
print("sessions", len(set(session)), "(expected 73; Supplementary Table 1 sums to 75)")
print("per subject (mapping js10..js27 -> 1..10 from cueAnalysis.m):")
print(pd.DataFrame({"subject": range(1, 11), "animal": bst3.RATS, "units": got, "expected": expected,
                    "sessions": per.sessions.values, "Supp. Table 1": per["sessions, Supp. Table 1"].values}).to_string(index=False))
print("all per-subject totals equal:", got == expected)
print("session duration (first to last cue onset), min: %.0f to %.0f" % (trials["session min"].min(), trials["session min"].max()))

**Result: reproduced.** 1812 units, 965 from females, 73 sessions, and the ten per-subject totals
equal the paper's. Supplementary Table 1 lists 75 sessions; the data set has one session fewer for
subject 5 (7 against 8) and for subject 10 (9 against 10), which accounts for the 73 of the Results.

## fig1e-regions

> The units spanned 21 brainstem regions ... with the largest counts in the paramedian raphe (236),
> ventral dorsal raphe (208) and dorsolateral periaqueductal gray (179).

In [ ]:
cnt = np.bincount(region, minlength=22)[1:]
t = pd.DataFrame({"region": bst3.REGIONS, "units": cnt,
                  "Source Data Fig. 3e left (sum)": [sum(r[i + 1] for r in sheets["Figure 3e left"][1:3]) for i in range(21)]})
print(t.to_string(index=False))
print("regions with units:", int((cnt > 0).sum()))
print("PMnR %d (236), DRV %d (208), DLPAG %d (179), VLPAG %d (147), DPG %d (145)" %
      (cnt[19], cnt[8], cnt[2], cnt[4], cnt[0]))

**Result: reproduced.** 21 regions hold units, and the five counts equal the expected values; every
region's count equals the Source Data Fig. 3e left sums.

## fig2b-cue-clusters

> K-means clustering of cue firing organized brainstem neurons into at least 21 functional clusters,
> of sizes from 24 to 219 (median 83).

k-means with 21 clusters on the 1812 x 42 cue matrix, k-means++ seeding, one replicate, 20 seeds
(the claims file's standard choice; the script uses MATLAB `rng(1)`, which cannot be reproduced
here). The best of the 20 (lowest summed distance) is a variant.

In [ ]:
seeds = np.arange(20)
km_cue = bst3.kmeans_sizes(X_cue, 21, seeds)
sz = np.array([[s.min(), s.max(), np.median(s)] for s, _, _ in km_cue])
print("per seed (min, max, median):")
print(pd.DataFrame(sz, columns=["min", "max", "median"], index=seeds).T.to_string())
med = np.median(sz, 0)
print("median over seeds: min %.1f (24 +- 2.4), max %.1f (219 +- 21.9), median %.1f (83 +- 8.3)" % tuple(med))
best = int(np.argmin([i for _, i, _ in km_cue]))
print("best of 20 (seed %d): sizes" % best, sorted(km_cue[best][0].tolist()))
print("L1 sizes (paper):", sorted(np.bincount(L1_cue)[1:].tolist()))
inertia_L1 = sum(((X_cue[L1_cue == k] - X_cue[L1_cue == k].mean(0)) ** 2).sum() for k in range(1, 22))
print(f"summed squared distance: paper's clustering {inertia_L1:.0f}; seeds {min(i for _, i, _ in km_cue):.0f} to {max(i for _, i, _ in km_cue):.0f}")
from sklearn.cluster import KMeans
msd = [KMeans(n_clusters=k, n_init=1, random_state=0).fit(X_cue).inertia_ / len(X_cue) for k in range(1, 26)]
print("mean squared distance for 1 to 25 clusters:", np.round(msd, 2).tolist())

**Result: reproduced.** Over 20 seeds the median smallest cluster is 24.0 (24 +- 2.4), the median
largest 218.5 (219 +- 21.9) and the median of the median sizes 75.0 (83 +- 8.3, so 74.7 to 91.3),
all three within tolerance, the last near its edge. Single seeds range from 14 to 38 (smallest), 171
to 274 (largest) and 63 to 93 (median). The paper's own clustering has a summed squared distance of
11219, inside the range of the 20 seeds (11135 to 11336). The mean squared distance falls smoothly
from 1 to 25 clusters, with no break at 21.

## fig2c-cue-pc1-ordered

> Across all 1812 neurons, the first principal component of cue firing (29.7% of firing variance)
> was ordered cue firing, danger > uncertainty > safety.

In [ ]:
coef_cue, expl_cue = bst3.pca(X_cue)
pc1 = coef_cue[:, 0] * np.sign(coef_cue[2:12, 0].mean())
w = {c: pc1[i * 14 + 2:i * 14 + 12].mean() for i, c in enumerate(["danger", "uncertainty", "safety"])}
print(f"PC1 {expl_cue[0]:.2f}% (29.7 +- 2.97), PC2 {expl_cue[1]:.2f}%, PC3 {expl_cue[2]:.2f}%")
print("mean PC1 weight over the 10 cue seconds:", {k: round(v, 4) for k, v in w.items()})
print("ordered danger > uncertainty > safety:", w["danger"] > w["uncertainty"] > w["safety"])
A = bst3.sd_matrix(sheets, "Figure 2c inset", 1, 42)
print(f"On the Source Data Fig. 2c inset matrix ({A.shape}): PC1 {bst3.pca(A)[1][0]:.2f}%")
hits, br, _ = bst3.match_rows(A, X_cue)
print("inset rows equal to a unit's cue row:", sum(len(h) > 0 for h in hits), "; median best r", np.median(br).round(3))
# the inset as the heat-plot version (baseline of the three cues' 2 pre-cue seconds subtracted, cueAnalysis.m)
base = (X_cue[:, 0:2].mean(1) + X_cue[:, 14:16].mean(1) + X_cue[:, 28:30].mean(1)) / 3
hits, br, _ = bst3.match_rows(A, X_cue - base[:, None])
print("inset rows equal to a unit's baseline-subtracted row:", sum(len(h) > 0 for h in hits))
print("\nPer animal (PC1 of that animal's units):")
for s_ in range(1, 11):
    c_, e_ = bst3.pca(X_cue[subject == s_])
    q = c_[:, 0] * np.sign(c_[2:12, 0].mean())
    w_ = [q[i * 14 + 2:i * 14 + 12].mean() for i in range(3)]
    print(f"  subject {s_} ({int((subject == s_).sum())} units): PC1 {e_[0]:.1f}%, mean cue weights D, U, S {np.round(w_, 3).tolist()}, "
          f"ordered {w_[0] > w_[1] > w_[2]}")

**Result: reproduced.** PC1 of the data set's 1812 x 42 cue matrix explains 29.73% of the variance
(29.7 +- 2.97), and its mean weights over the 10 cue seconds are ordered danger 0.266 > uncertainty
0.128 > safety 0.006. The 47.09% on the Source Data inset comes from another matrix: its 1812 rows
equal the units' cue rows with the mean of the three cues' two pre-cue seconds subtracted (the heat
plot of `cueAnalysis.m`), not the rows PCA was run on. Per animal, PC1 explains 19.2 to 43.0%, and
its cue weights are ordered danger > uncertainty > safety in 9 of 10 animals (subject 9: uncertainty
0.139, safety 0.140).

## fig2c-pc1-contribution

> PC1 firing information originated from nine clusters (k1 to k9), a minority of neurons (505 of
> 1812, 27.9%), with k1 and k2 contributing most per unit.

Per cluster, its rows of the cue matrix shuffled 1000 times, PCA each time; the drop of PC1 percent
from the intact value, averaged, divided by the cluster size (cueAnalysis.m). Readings s-a (each
row across its columns; the script's `Shuffle(..., 2)`) and s-b (the block's entries together).

In [ ]:
rng = np.random.default_rng(3)
sd2c = np.array([r[1] for r in sheets["Figure 2c"][1:22]], float)
contrib = {}
for mode in ("s-a", "s-b"):
    contrib[mode] = bst3.pc1_contribution(X_cue, L1_cue, range(1, 22), rng, 1000, mode)
t = pd.DataFrame({"cluster": range(1, 22), "size": np.bincount(L1_cue)[1:],
                  "s-a": [contrib["s-a"][k] for k in range(1, 22)],
                  "s-b": [contrib["s-b"][k] for k in range(1, 22)], "Source Data": sd2c})
print(t.round(4).to_string(index=False))
for mode in ("s-a", "s-b"):
    v = np.array([contrib[mode][k] for k in range(1, 22)])
    top9 = set(np.argsort(-v)[:9] + 1)
    top2 = set(np.argsort(-v)[:2] + 1)
    print(f"{mode}: nine largest {sorted(top9)}; k1-k9 out of place: {len(set(range(1, 10)) - top9)}; two largest {sorted(top2)}; "
          f"k1 {v[0]:.4f} (0.121 +- 0.012), k2 {v[1]:.4f} (0.113 +- 0.011); max abs diff from Source Data {np.abs(v - sd2c).max():.4f}")
print("units in k1-k9:", int(np.isin(L1_cue, range(1, 10)).sum()), "of", len(L1_cue))
print("share of each animal's units in k1-k9:", {s_: round(float(np.isin(L1_cue[subject == s_], range(1, 10)).mean()), 3) for s_ in range(1, 11)})

In [ ]:
# L2: each of the 20 k-means solutions, its clusters numbered by PC1 contribution per unit (s-a).
# The claims file's standard reports the median over the 20 seeds; the best of 20 is a variant.
L2_cue_seeds, l2rows = [], []
for s_ in seeds:
    raw = km_cue[s_][2] + 1
    c2 = bst3.pc1_contribution(X_cue, raw, range(1, 22), np.random.default_rng(100 + s_), 1000, "s-a")
    rank = sorted(c2, key=lambda k: -c2[k])
    lab = np.zeros_like(raw)
    for new, old in enumerate(rank, 1):
        lab[raw == old] = new
    L2_cue_seeds.append(lab)
    sub = np.isin(lab, range(1, 10))
    l2rows.append({"seed": s_, "nine largest hold": int(sub.sum()),
                   "shared with L1 k1-k9": int((sub & np.isin(L1_cue, range(1, 10))).sum()),
                   "k1 size": int((lab == 1).sum()), "k2 size": int((lab == 2).sum()),
                   "k1 per unit": round(c2[rank[0]], 4), "k2 per unit": round(c2[rank[1]], 4)})
l2 = pd.DataFrame(l2rows)
print(l2.to_string(index=False))
tot = l2["nine largest hold"]
print(f"median over the 20 seeds of the units in the nine largest: {tot.median():.1f} (505 +- 50.5); range {tot.min()} to {tot.max()}; "
      f"within tolerance in {int(((tot - 505).abs() <= 50.5).sum())} of 20 seeds")
L2_cue = L2_cue_seeds[best]
print(f"best of 20 (seed {best}), the L2 variant used below: the nine largest hold {int(np.isin(L2_cue, range(1, 10)).sum())} units, "
      f"{int((np.isin(L1_cue, range(1, 10)) & np.isin(L2_cue, range(1, 10))).sum())} of them in L1's k1-k9; sizes in contribution order:",
      [int((L2_cue == k).sum()) for k in range(1, 22)])

**Result: reproduced.** Under L1 the nine largest contributions per unit are those of k1 to k9 (505
units), k1 and k2 are the two largest, and k1 (0.1214) and k2 (0.1129) are within tolerance under
reading s-a, the script's shuffle; s-b gives 0.1217 and 0.1127. Reading s-a gives every cluster's
Source Data value to within 0.0003. Under L2 (this attempt's own k-means, numbered by contribution)
the nine largest clusters hold a median of 481 units over the 20 seeds, within 505 +- 50.5, though
single seeds range from 360 to 590 and 13 of 20 are within tolerance. The best of the 20 solutions,
the L2 variant used in later sections, gives 411.

## fig2d-danger-latency

> Clusters k1 to k9 showed shorter firing latencies after danger onset, and across all 21 clusters
> danger firing latency correlated negatively with the magnitude of PC1 contribution (R2 = 0.41,
> p = 0.0017).

Per unit, `findchangepts` (one change in mean) on its danger firing over the 10-s cue in 1-ms bins
(Supplementary Methods, Danger firing latency). Baseline subtraction and z-scoring are affine maps
of the summed spike count, and the change point of a mean is unchanged by them, so reading l-a is
the change point of the summed 1-ms spike counts over the four danger trials. Reading l-b uses the
cube's 250-ms z values repeated into 1-ms bins. A unit with no change point (no danger spikes) has
no latency. A variant outside the criterion searches from 2 s before onset and keeps latencies
after onset. The table counts, per reading, the latencies equal to a Source Data value of the same
cluster. None of the paper's scripts computes the latency (data check above).

In [ ]:
ms1 = units["ms1"][ix]                       # summed danger-trial spikes, 1-ms bins from -2 s to +10 s
sd2d = pd.DataFrame([r for r in sheets["Figure 2d"][1:] if r[0] is not None], columns=["cluster", "latency"])
print("Source Data Fig. 2d lists", len(sd2d), "units; units with no danger spikes in the cue:", int((ms1[:, 2000:].sum(1) == 0).sum()))
z250 = cube["fire/z/ms250/interval"][:, :, 0:4].mean(2)[:, 40:80]     # danger, 40 cue bins of 250 ms
lat_readings = {
    "l-a (10-s cue, 1-ms spikes)": np.array([bst3.change_point(x) or np.nan for x in ms1[:, 2000:]], float),
    "l-b (10-s cue, 250-ms z repeated)": np.array([bst3.change_point(np.repeat(x, 250)) or np.nan for x in z250], float),
}
lat12 = np.array([(bst3.change_point(x) or np.nan) - 2000 for x in ms1], float)
lat_readings["variant: 12-s window from 2 s before onset, latency > 0 kept"] = np.where(lat12 > 0, lat12, np.nan)
sd_sets = {k: Counter(sd2d[sd2d.cluster == k].latency.astype(int)) for k in range(1, 22)}
cvec = np.array([contrib["s-a"][k] for k in range(1, 22)])
size = np.bincount(L1_cue)[1:]
sd_cl = sd2d.groupby("cluster").latency.mean().reindex(range(1, 22)).values
rows = []
for name, lat in lat_readings.items():
    shared = 0
    for k in range(1, 22):
        mine = Counter(lat[(L1_cue == k) & np.isfinite(lat)].astype(int))
        shared += sum((mine & sd_sets[k]).values())
    cl = np.array([np.nanmean(lat[L1_cue == k]) for k in range(1, 22)])
    ra, pa = stats.pearsonr(cvec, cl)
    rb, pb = stats.pearsonr(cvec * size, cl)
    sub = np.isin(L1_cue, range(1, 10))
    rows.append({"reading": name, "units with latency": int(np.isfinite(lat).sum()), "equal to a Source Data value": shared,
                 "R2 c-a": round(ra * ra, 3), "r c-a": round(ra, 3), "p c-a": pa, "R2 c-b": round(rb * rb, 3), "p c-b": pb,
                 "k1-k9 mean ms": round(np.nanmean(lat[sub])), "k10-k21 mean ms": round(np.nanmean(lat[~sub])),
                 "k1 mean": round(cl[0]), "k2 mean": round(cl[1])})
for name, cl in (("Source Data latencies", sd_cl),):
    ra, pa = stats.pearsonr(cvec, cl)
    rb, pb = stats.pearsonr(cvec * size, cl)
    rows.append({"reading": name, "units with latency": len(sd2d), "R2 c-a": round(ra * ra, 3), "r c-a": round(ra, 3),
                 "p c-a": pa, "R2 c-b": round(rb * rb, 3), "p c-b": pb, "k1 mean": round(cl[0]), "k2 mean": round(cl[1])})
print(pd.DataFrame(rows).to_string(index=False))
lat = lat_readings["l-a (10-s cue, 1-ms spikes)"]
print("\nPer cluster (l-a; variant; Source Data):")
print(pd.DataFrame({"cluster": range(1, 22),
                    "l-a n": [int(np.isfinite(lat[L1_cue == k]).sum()) for k in range(1, 22)],
                    "l-a mean": [round(np.nanmean(lat[L1_cue == k])) for k in range(1, 22)],
                    "variant n": [int(np.isfinite(lat_readings["variant: 12-s window from 2 s before onset, latency > 0 kept"][L1_cue == k]).sum()) for k in range(1, 22)],
                    "variant mean": [round(np.nanmean(lat_readings["variant: 12-s window from 2 s before onset, latency > 0 kept"][L1_cue == k])) for k in range(1, 22)],
                    "Source Data n": sd2d.groupby("cluster").size().reindex(range(1, 22)).values,
                    "Source Data mean": sd_cl.round(0)}).to_string(index=False))

**Result: not reproduced.** Under the claims file's readings (latency found within the 10-s cue),
the correlation across the 21 clusters between mean danger latency and PC1 contribution per unit is
r = -0.260, R2 = 0.067, p = 0.26 (l-a), and R2 = 0.065, p = 0.26 (l-b); with the cluster total
(c-b) R2 is 0.122 and 0.104, p >= 0.12. Units of k1 to k9 do have shorter latencies (3571 against
4590 ms, l-a). The latencies differ from the paper's: l-a gives 1778 units a latency where Source
Data lists 1498, and k1's mean is 4046 ms here and 182 ms in Source Data. Searching for the change
point from 2 s before cue onset, and keeping latencies after onset (a variant outside the
criterion), gives 1458 units with latency, k1 327 ms and k2 307 ms, and R2 = 0.640 (c-a,
p = 1.4e-05) and 0.661 (c-b). This variant does not reproduce the Source Data latencies either: 413
of its values equal a Source Data value in the same cluster, fewer than the 513 of l-a, and its k1
mean is 327 ms against 182 ms. How the paper's latencies were computed is therefore open. On the
paper's own Source Data latencies the correlation is R2 = 0.508 (c-a) and 0.394 (c-b).

## fig2e-cue-subnetwork

> Clusters k1 to k9 separated from the rest by cluster-cluster cue firing correlations ... with k3
> the exception showing weak correlations with its fellow clusters.

In [ ]:
z_types = {c: m for c, m in zip(["danger", "uncertainty", "safety"], bst3.type_means(zs1, ["danger", "uncertainty", "safety"]))}
def cluster_profiles(cols):
    P = np.hstack([z_types[c][:, cols] for c in ["danger", "uncertainty", "safety"]])
    return np.array([P[L1_cue == k].mean(0) for k in range(1, 22)])
sub_idx, sup_idx = np.arange(9), np.arange(9, 21)
for name, cols in (("42 columns (Source Data, 14 per cue)", slice(8, 22)), ("36 columns (Methods, 12 per cue)", slice(8, 20)),
                   ("120 columns (script's full 40-s window)", slice(0, 40))):
    R = np.corrcoef(cluster_profiles(cols))
    for power, lab in ((1, "r"), (2, "r squared")):
        Q = R ** power
        within = Q[np.ix_(sub_idx, sub_idx)][np.triu_indices(9, 1)].mean()
        between = Q[np.ix_(sub_idx, sup_idx)].mean()
        k3 = [(Q[i, [j for j in sub_idx if j != i]]).mean() for i in sub_idx]
        print(f"{name}, {lab}: within k1-k9 {within:.3f}, k1-k9 with k10-k21 {between:.3f}; "
              f"mean with the other eight per cluster {np.round(k3, 3).tolist()}; lowest k{int(np.argmin(k3)) + 1}")
sd2e = bst3.sd_matrix(sheets, "Figure 2e", 1, 42)
print("Source Data Fig. 2e rows equal the L1 cluster means:", np.allclose(sd2e, cluster_profiles(slice(8, 22)), atol=1e-9))

**Result: reproduced.** With r squared, the measure the Methods name ("Pearson's correlation
coefficient (R2)") and the script plots, the mean within k1 to k9 (0.445, 42 columns; 0.465, 36
columns) exceeds the mean between k1 to k9 and k10 to k21 (0.184; 0.194), and k3 has the lowest mean
with the other eight subnetwork clusters (0.225; 0.229). With r the within mean also exceeds the
between mean (0.129 against 0.068), but k8 is lowest (-0.516), as k7 and k8 decrease firing while
the other subnetwork clusters increase it, and k3 is not. The Source Data Fig. 2e rows equal the
L1 cluster means.

## fig2f-cue-hubs

> K1 and k2 were hubs of the cue subnetwork ... single-unit firing that correlated most strongly
> with the mean cue firing of their fellow subnetwork clusters.

Per unit of k1 to k9, its danger firing over the 14 1-s intervals correlated with the mean danger
firing of each other subnetwork cluster, averaged. The Source Data column is headed "R2", and the
script's outcome analysis squares the coefficients, so r and r squared are both computed, and the
42-value row (all three cues) is a variant; the one that equals Source Data Fig. 2f per unit shows
which the paper used.

In [ ]:
sd2f = pd.DataFrame([r for r in sheets["Figure 2f"][1:] if r[0] is not None], columns=["cluster", "R2"])
dz = z_types["danger"][:, 8:22]
means_d = {k: dz[L1_cue == k].mean(0) for k in range(1, 10)}
means_all = {k: X_cue[L1_cue == k].mean(0) for k in range(1, 10)}
variants = {}
for name, rows_, means in (("danger 14", dz, means_d), ("all 42", X_cue, means_all)):
    for sq in (False, True):
        v = np.full(len(ids), np.nan)
        for u in np.flatnonzero(np.isin(L1_cue, range(1, 10))):
            k = L1_cue[u]
            v[u] = bst3.unit_cluster_corr(rows_[u], [means[j] for j in range(1, 10) if j != k], sq)
        variants[f"{name}, {'r squared' if sq else 'r'}"] = v
rows = []
for name, v in variants.items():
    m = [np.nanmean(v[L1_cue == k]) for k in range(1, 10)]
    same = sum(np.isclose(np.sort(v[L1_cue == k]), np.sort(sd2f[sd2f.cluster == k].R2.values), atol=1e-6).all()
               for k in range(1, 10))
    rows.append({"measure": name, **{f"k{k}": round(m[k - 1], 3) for k in range(1, 10)}, "clusters equal to Source Data": same})
rows.append({"measure": "Source Data", **{f"k{k}": round(sd2f[sd2f.cluster == k].R2.mean(), 3) for k in range(1, 10)}})
print(pd.DataFrame(rows).to_string(index=False))
v = variants["danger 14, r squared"]
for thr_n in (7, 8):
    print(f"\nthreshold 0.05/{thr_n} = {0.05 / thr_n:.4f}")
    for hub in (1, 2):
        out = []
        for k in range(3, 10):
            t_ = stats.ttest_ind(v[L1_cue == hub], v[L1_cue == k])
            lv = stats.levene(v[L1_cue == hub], v[L1_cue == k], center="mean")
            out.append(f"k{k}: t={t_.statistic:.2f} p={t_.pvalue:.2g}{'*' if t_.pvalue < 0.05 / thr_n and t_.statistic > 0 else ''}"
                       f" (Levene p={lv.pvalue:.2g})")
        print(f"k{hub} vs", "; ".join(out))

In [ ]:
# Tests for every measure: does each hub exceed each of k3-k9 (t-test, one direction) at 0.05/7 and 0.05/8
for name, v in variants.items():
    out = []
    for hub in (1, 2):
        ps = [stats.ttest_ind(v[L1_cue == hub], v[L1_cue == k]) for k in range(3, 10)]
        out.append(f"k{hub} above {sum(t.pvalue < 0.05 / 7 and t.statistic > 0 for t in ps)}/7 at 0.05/7, "
                   f"{sum(t.pvalue < 0.05 / 8 and t.statistic > 0 for t in ps)}/7 at 0.05/8")
    m = [np.nanmean(v[L1_cue == k]) for k in range(1, 10)]
    print(f"{name}: two highest k{int(np.argsort(m)[-1]) + 1}, k{int(np.argsort(m)[-2]) + 1}; " + "; ".join(out))
# The same with r (the claims file's wording, "coefficients averaged")
v = variants["danger 14, r"]
for hub in (1, 2):
    ps = [stats.ttest_ind(v[L1_cue == hub], v[L1_cue == k]) for k in range(3, 10)]
    print(f"r: k{hub} mean {np.nanmean(v[L1_cue == hub]):.3f}; exceeds k3-k9 at 0.05/7: "
          f"{sum(t.pvalue < 0.05 / 7 and t.statistic > 0 for t in ps)} of 7; at 0.05/8: {sum(t.pvalue < 0.05 / 8 and t.statistic > 0 for t in ps)} of 7")
v = variants["danger 14, r squared"]
for hub in (1, 2):
    ps = [stats.ttest_ind(v[L1_cue == hub], v[L1_cue == k]) for k in range(3, 10)]
    print(f"r squared: k{hub} mean {np.nanmean(v[L1_cue == hub]):.3f}; exceeds k3-k9 at 0.05/7: "
          f"{sum(t.pvalue < 0.05 / 7 and t.statistic > 0 for t in ps)} of 7; at 0.05/8: {sum(t.pvalue < 0.05 / 8 and t.statistic > 0 for t in ps)} of 7")
# Variant: the Methods' eight-cluster subnetwork (k1-k8)
v8 = np.full(len(ids), np.nan)
for u in np.flatnonzero(np.isin(L1_cue, range(1, 9))):
    k = L1_cue[u]
    v8[u] = bst3.unit_cluster_corr(dz[u], [means_d[j] for j in range(1, 9) if j != k], True)
print("k1-k8 subnetwork, r squared means:", [round(np.nanmean(v8[L1_cue == k]), 3) for k in range(1, 9)])

**Result: partially reproduced.** With the Methods' measure (danger firing, 14 intervals) as r
squared, the Source Data column's "R2", k2 (0.503) and k1 (0.462) have the two highest means, both
within tolerance (0.466 +- 0.047 and 0.424 +- 0.042), but not every t-test passes: k1 does not exceed
k3 (p = 0.42) or k4 (p = 0.12), and k2 does not exceed k3 (p = 0.014) at 0.05/7 or 0.05/8. With r,
k1 and k3 are the two highest. The Source Data per-unit values equal, in all nine clusters, r squared
computed on the full 42-value cue rows (all three cues), not on danger firing; with that measure
k1 (0.424) and k2 (0.466) are the two highest and each exceeds all seven of k3 to k9 at both
thresholds.

## fig2g-cue-anatomy

> Neurons of every cue cluster were found in at least six brainstem regions, and subnetwork neurons,
> including the k1 and k2 hubs, were concentrated in the deep layer of the superior colliculus and
> the subdivisions of the periaqueductal gray.

In [ ]:
nreg = [len(set(region[L1_cue == k])) for k in range(1, 22)]
print("regions per cue cluster:", nreg, "; minimum", min(nreg))
sc_pag = np.isin(region, [1, 2, 3, 4, 5])
sub = np.isin(L1_cue, range(1, 10))
hub = np.isin(L1_cue, [1, 2])
print(f"share in DPG, DpWh, DLPAG, LPAG, VLPAG: subnetwork {sc_pag[sub].mean():.3f} ({sc_pag[sub].sum()}/{sub.sum()}), "
      f"supranetwork {sc_pag[~sub].mean():.3f} ({sc_pag[~sub].sum()}/{(~sub).sum()}), k1 and k2 {sc_pag[hub].mean():.3f} ({sc_pag[hub].sum()}/{hub.sum()})")
tab = pd.crosstab(sub, sc_pag)
print("chi-square, subnetwork x SC/PAG (shown, not the criterion):", stats.chi2_contingency(tab)[:2])

**Result: reproduced.** Every cue cluster spans at least six regions (k3 has six, the fewest). The
share of units in DPG, DpWh, DLPAG, LPAG and VLPAG is 0.436 (220/505) in the subnetwork, 0.612
(41/67) in k1 and k2, and 0.340 (445/1307) in the supranetwork.

## fig3a-cluster-signals

> Linear regression revealed distinct threat probability and fear behaviour signals across the 21
> cue clusters, varying in time, with greatest signaling at onset (k5), at offset (k15), sustained
> over the cue (k10), or peaking mid cue (k6).

Regression per cluster and 1-s interval; readings r-a (the script's: cluster mean firing, mean
member suppression ratio) and r-b (per unit, betas averaged), each with (i-a) and without (i-b) an
intercept. Signal per interval: the larger absolute beta. Interval indices: 0-1 pre-cue, 2-11 the
cue seconds, 12-13 after.

In [ ]:
sd3b = bst3.sd_matrix(sheets, "Figure 3b middle", 2, 28)
betas = {}
for mode in ("r-a", "r-b"):
    for ic, iname in ((True, "i-a"), (False, "i-b")):
        betas[(mode, iname)] = bst3.cue_betas(zs1, sr, L1_cue, range(1, 22), mode, ic)
for key, B in betas.items():
    print(key, "max abs difference from Source Data Fig. 3b middle:", round(float(np.abs(B - sd3b).max()), 6))
def shapes(B):
    s = np.maximum(np.abs(B[:, :14]), np.abs(B[:, 14:]))
    k5, k15, k10, k6 = s[4], s[14], s[9], s[5]
    return {"onset k5": int(np.argmax(k5)) == 2, "offset k15": int(np.argmax(k15)) in (11, 12),
            "sustained k10": bool((k10[2:12] >= 0.5 * k10.max()).all()),
            "mid-cue k6": int(np.argmax(k6)) in range(4, 10) and k6[2] <= 0.5 * k6.max() and k6[11] <= 0.5 * k6.max()}, s
for key, B in betas.items():
    ok, s = shapes(B)
    print(key, ok, "; held:", sum(ok.values()))
ok, s = shapes(betas[("r-a", "i-a")])
print("\nr-a, i-a signal per interval (larger |beta|):")
print(pd.DataFrame(s[[4, 14, 9, 5]], index=["k5", "k15", "k10", "k6"]).round(2).to_string())

**Result: partially reproduced.** Under the script's regression (r-a, i-a), whose betas equal the
Source Data Fig. 3b matrix exactly, two of the four shapes hold: k5 signals most in the first cue
second and k15 at the first second after the cue. K10 is not sustained by the criterion's rule (its
first cue second, 0.77, is below half its maximum, 1.69), and k6 peaks in cue second 5 (1.02) but its
first cue second (0.60) is above half of that. The other three readings give the same two of four.

In [ ]:
def orient_cue(c):
    pc1 = c[:, 0] * np.sign(c[16:26, 0].mean())
    pc2 = c[:, 1] * np.sign(c[2:12, 1].mean())
    return pc1, pc2
def cue_shape(pc1, pc2):
    beh1, prob1 = pc1[14:], pc1[:14]
    beh2, prob2 = pc2[14:], pc2[:14]
    return {"PC1 |beh| > |prob| over cue": np.abs(beh1[2:12]).mean() > np.abs(prob1[2:12]).mean(),
            "PC1 beh max in cue s 3-8": int(np.argmax(beh1)) in range(4, 10),
            "PC1 mean prob over cue < 0": prob1[2:12].mean() < 0,
            "PC2 |prob| > |beh| over cue": np.abs(prob2[2:12]).mean() > np.abs(beh2[2:12]).mean(),
            "PC2 prob max in cue s 1-3": int(np.argmax(prob2)) in range(2, 5)}

## fig3b-signal-pca

> PCA of cluster beta weights showed PC1 (62.7% of signaling variance) to be a sustained behaviour
> signal peaking mid cue with lesser, opposing threat probability signaling, and PC2 (26.12%) a
> sustained threat probability signal peaking early in the cue with lesser behaviour signaling.

PC1 oriented so its mean behaviour weight over the 10 cue seconds is positive, PC2 so its mean
probability weight is positive. The weights are also shown with the script's plotting baseline
(the mean of the two pre-cue seconds) subtracted, as a variant.

In [ ]:
B_cue = betas[("r-a", "i-a")]
rows = []
mats = {"Source Data Fig. 3b middle": sd3b, "recomputed, L1 (r-a, i-a)": B_cue}
for key in (("r-b", "i-a"), ("r-a", "i-b"), ("r-b", "i-b")):
    mats[f"recomputed, L1 {key}"] = betas[key]
mats["recomputed, L2 (r-a, i-a)"] = bst3.cue_betas(zs1, sr, L2_cue, range(1, 22), "r-a", True)
pcs_cue = {}
for name, M in mats.items():
    c, e = bst3.pca(M)
    p1, p2 = orient_cue(c)
    pcs_cue[name] = (p1, p2, e)
    sh = cue_shape(p1, p2)
    rows.append({"matrix": name, "PC1 %": round(e[0], 2), "PC2 %": round(e[1], 2), **sh})
print(pd.DataFrame(rows).to_string(index=False))
p1, p2, e = pcs_cue["recomputed, L1 (r-a, i-a)"]
print("\nPC1 probability weights:", p1[:14].round(3).tolist())
print("PC1 behaviour weights:  ", p1[14:].round(3).tolist())
print("PC2 probability weights:", p2[:14].round(3).tolist())
print("PC2 behaviour weights:  ", p2[14:].round(3).tolist())
bl = lambda v: np.concatenate([v[:14] - v[:2].mean(), v[14:] - v[14:16].mean()])
print("with the pre-cue baseline subtracted:", cue_shape(bl(p1), bl(p2)))

**Result: reproduced.** PCA of the recomputed 21 x 28 beta matrix (equal to Source Data Fig. 3b
middle) gives PC1 62.73% and PC2 26.21%, within tolerance, and all five shape conditions hold, also
with the pre-cue baseline subtracted. Per-unit regression (r-b) and no intercept (i-b) give 72.00 and
15.79%, 57.07 and 27.94%, and 66.16 and 21.37%; the paper's numbers come from the script's reading.
L2 gives 63.08 and 25.04% with all shapes.

## fig3c-subnetwork-lesion and fig3d-supranetwork-lesion

> Lesioning (shuffling) the cue subnetwork left the sustained behaviour signal intact but reduced
> threat probability signaling, most at cue onset ...

> Lesioning the cue supranetwork diminished the sustained behaviour signal and turned the sustained
> threat probability signal into a dynamic probability-to-behaviour signal ...

Lesion as in `cueRegression.m`: each shuffled cluster's trial rows permuted across the 14 intervals
before regression, betas averaged over 1000 iterations (s-c), then one PCA (v-a). Variants: s-a and
s-b on the intact betas, and v-b (PCA per iteration on s-a, sign-aligned to the intact PCs).

In [ ]:
rng = np.random.default_rng(5)
lesion = {}
for name, shuffled in (("sub", set(range(1, 10))), ("supra", set(range(10, 22)))):
    lesion[(name, "s-c")] = bst3.cue_lesion_betas(zs1, sr, L1_cue, range(1, 22), shuffled, rng, 1000)
    rows_ = np.array(sorted(shuffled)) - 1
    lesion[(name, "s-a")] = bst3.beta_shuffle_average(B_cue, rows_, rng, 1000, "s-a")
    lesion[(name, "s-b")] = bst3.beta_shuffle_average(B_cue, rows_, rng, 1000, "s-b")
sd3c, sd3d = bst3.sd_matrix(sheets, "Figure 3c middle", 2, 28), bst3.sd_matrix(sheets, "Figure 3d middle", 2, 28)
print("max abs difference from Source Data: sub s-c %.3f, supra s-c %.3f" %
      (np.abs(lesion[("sub", "s-c")] - sd3c).max(), np.abs(lesion[("supra", "s-c")] - sd3d).max()))
ip1, ip2, ie = pcs_cue["recomputed, L1 (r-a, i-a)"]
rows = []
les_pcs = {}
for key, M in list(lesion.items()) + [(("sub", "Source Data"), sd3c), (("supra", "Source Data"), sd3d)]:
    c, e = bst3.pca(M)
    p1, p2 = orient_cue(c)
    les_pcs[key] = (p1, p2, e)
    rows.append({"lesion": key[0], "reading": key[1], "PC1 %": round(e[0], 2), "PC2 %": round(e[1], 2),
                 "PC1 mean beh over cue": round(p1[16:26].mean(), 3), "PC2 prob, first cue s": round(p2[2], 3),
                 "PC2 |prob| peak s": int(np.argmax(np.abs(p2[2:12]))) + 1, "PC2 |beh| peak s": int(np.argmax(np.abs(p2[16:26]))) + 1})
rows.append({"lesion": "intact", "reading": "r-a, i-a", "PC1 %": round(ie[0], 2), "PC2 %": round(ie[1], 2),
             "PC1 mean beh over cue": round(ip1[16:26].mean(), 3), "PC2 prob, first cue s": round(ip2[2], 3),
             "PC2 |prob| peak s": int(np.argmax(np.abs(ip2[2:12]))) + 1, "PC2 |beh| peak s": int(np.argmax(np.abs(ip2[16:26]))) + 1})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# v-b: PCA per iteration, sign-aligned to the intact PCs, averaged; on the script's lesion (s-c, one
# iteration of cueRegression.m's shuffle each time) and on the beta shuffle s-a
ref = np.column_stack([ip1, ip2])
for name, shuffled in (("sub", range(1, 10)), ("supra", range(10, 22))):
    rws = np.array(list(shuffled)) - 1
    def make_sc(i, sh=set(shuffled), g=np.random.default_rng(6)):
        return bst3.cue_lesion_betas(zs1, sr, L1_cue, range(1, 22), sh, g, 1)
    def make_sa(i, rws=rws, g=np.random.default_rng(6)):
        C = B_cue.copy()
        C[rws] = bst3.shuffle_rows(B_cue[rws], g)
        return C
    for mode, make in (("s-c", make_sc), ("s-a", make_sa)):
        W, P = bst3.pca_per_iteration(make, 1000, ref)
        print(f"v-b {name} {mode}: PC1 {P[0]:.2f}%, PC2 {P[1]:.2f}%; PC1 mean beh over cue {W[16:26, 0].mean():.3f}; PC2 prob first cue s {W[2, 1]:.3f}")
for name in ("s-c", "s-a", "s-b", "Source Data"):
    s1, s2, _ = les_pcs[("sub", name)]
    u1, u2, _ = les_pcs[("supra", name)]
    c_ = {"3c PC1 beh >= 75% intact": s1[16:26].mean() >= 0.75 * ip1[16:26].mean(),
          "3c PC1 beh > supra-lesion's": s1[16:26].mean() > u1[16:26].mean(),
          "3c PC2 prob at first cue s < intact": s2[2] < ip2[2],
          "3d PC1 beh < intact": u1[16:26].mean() < ip1[16:26].mean(),
          "3d PC2 |prob| peak in first half, |beh| in second": (int(np.argmax(np.abs(u2[2:12]))) < 5) and (int(np.argmax(np.abs(u2[16:26]))) >= 5)}
    print(name, c_)

In [ ]:
# Stability of s-c: ten further runs of 1000 iterations each, with their own generators
st = []
for s_ in range(10):
    r_ = {}
    for name, shuffled in (("sub", set(range(1, 10))), ("supra", set(range(10, 22)))):
        M = bst3.cue_lesion_betas(zs1, sr, L1_cue, range(1, 22), shuffled, np.random.default_rng(200 + s_), 1000)
        p1, p2 = orient_cue(bst3.pca(M)[0])
        r_[f"{name} PC1 beh"], r_[f"{name} PC2 prob 1st s"] = p1[16:26].mean(), p2[2]
    st.append(r_)
st = pd.DataFrame(st)
print(st.agg(["mean", "std", "min", "max"]).round(4).to_string())
print("runs in which the subnetwork lesion's PC1 behaviour weight is above the supranetwork lesion's:",
      int((st["sub PC1 beh"] > st["supra PC1 beh"]).sum()), "of 10")
# The criteria compare unit-norm PC weights; loadings (weight x square root of the PC's variance)
# also carry the size of each PC, shown as a variant
def loadings(M):
    c, e = bst3.pca(M)
    lat = e / 100 * M.var(0, ddof=1).sum()
    p1, p2 = orient_cue(c)
    return p1 * np.sqrt(lat[0]), p2 * np.sqrt(lat[1])
for key, M in (("intact", B_cue), ("sub s-c", lesion[("sub", "s-c")]), ("supra s-c", lesion[("supra", "s-c")])):
    l1, l2_ = loadings(M)
    print(f"loadings, {key}: PC1 mean beh over cue {l1[16:26].mean():.3f}; PC2 prob first cue s {l2_[2]:.3f}")

**Result: fig3c-subnetwork-lesion partially reproduced; fig3d-supranetwork-lesion reproduced.** With
the script's lesion (s-c), whose beta matrices differ from Source Data Fig. 3c and 3d by at most
0.114 and give the same percentages (67.14 and 26.00%, 61.12 and 29.42%; Source Data 67.14 and
26.01%, 61.11 and 29.46%), the subnetwork lesion keeps PC1's mean behaviour weight over the cue at
0.264 (intact 0.274, so 96%) and lowers PC2's probability weight in the first cue second from 0.268
to 0.201, but that behaviour weight is not above the one after the supranetwork lesion (0.268): one
of the criterion's two parts fails, on the paper's own matrices too. The difference is small but
stable: over ten further runs of 1000 iterations the two weights average 0.2641 and 0.2678
(standard deviation 0.0001 each), and the subnetwork lesion's is above in none of the ten. PCA per
iteration on s-c (v-b) gives the same order (0.263 against 0.266). The supranetwork lesion lowers
PC1's behaviour weight (0.268 against 0.274, a small drop), and its PC2 has its probability peak in
the first cue second and its behaviour peak in the tenth. Shuffling betas (s-a, s-b) meets every
part of fig3c and fails the PC2 part of fig3d.

The criteria compare unit-norm PC weights, which carry a PC's shape and not its size. With loadings
(a variant outside the criteria), the subnetwork lesion's mean PC1 behaviour loading (0.709) is
above the supranetwork lesion's (0.680) and is 94% of the intact 0.756, and its PC2 probability
loading in the first cue second falls from 0.479 to 0.335, so every part of fig3c would hold; the
supranetwork lesion's PC1 behaviour loading is also below the intact one. Whether the criteria
should compare loadings is asked of a reader.

## fig3e-network-anatomy

> Subnetwork and supranetwork neurons were distributed throughout the brainstem, with subtle biases
> only for the subnetwork, to which the deep layer of the superior colliculus and the lateral
> periaqueductal gray contributed more.

In [ ]:
d3e, a, b = bst3.differential(region, np.isin(L1_cue, range(1, 10)), np.isin(L1_cue, range(10, 22)))
sd3e = np.array(sheets["Figure 3e right"][1][1:22], float)
t = pd.DataFrame({"region": bst3.REGIONS, "k1-k9": a, "k10-k21": b, "differential": d3e.round(2), "Source Data": sd3e.round(2)})
print(t.to_string(index=False))
top = np.argsort(-d3e)[:4]
print("four largest:", [bst3.REGIONS[i] for i in top], "; DPG %.2f (5.93 +- 0.59), LPAG %.2f (7.41 +- 0.74)" % (d3e[0], d3e[3]))
# Variant: the Methods' wording, region proportion in the subnetwork minus the overall subnetwork proportion
var = np.array([np.isin(L1_cue[region == r], range(1, 10)).mean() for r in range(1, 22)]) - np.isin(L1_cue, range(1, 10)).mean()
print("variant (share of region in subnetwork minus 505/1812):", dict(zip(bst3.REGIONS, (100 * var).round(1))))
dL2, _, _ = bst3.differential(region, np.isin(L2_cue, range(1, 10)), np.isin(L2_cue, range(10, 22)))
print("L2: two largest", [bst3.REGIONS[i] for i in np.argsort(-dL2)[:2]], np.sort(dL2)[::-1][:4].round(2))

**Result: reproduced.** LPAG (7.41) and DPG (5.93) have the two largest differentials, each equal to
the Source Data value, followed by DRD (3.13) and PMnR (2.53); every region equals Source Data Fig. 3e
right. Under L2 LPAG and DPG are also the two largest (9.54 and 6.96).

## fig4b-outcome-clusters

> K-means clustering of mean post-shock firing for the four trial types organized brainstem
> neurons into at least 11 functional clusters, of sizes from 46 to 322 (median 169).

In [ ]:
km_out = bst3.kmeans_sizes(X_out, 11, seeds)
sz = np.array([[s.min(), s.max(), np.median(s)] for s, _, _ in km_out])
print(pd.DataFrame(sz, columns=["min", "max", "median"], index=seeds).T.to_string())
med = np.median(sz, 0)
print("median over seeds: min %.1f (46 +- 4.6), max %.1f (322 +- 32.2), median %.1f (169 +- 16.9)" % tuple(med))
best_o = int(np.argmin([i for _, i, _ in km_out]))
print("best of 20 (seed %d):" % best_o, sorted(km_out[best_o][0].tolist()), "; paper:", sorted(np.bincount(L1_out)[1:].tolist()))
msd = [KMeans(n_clusters=k, n_init=1, random_state=0).fit(X_out).inertia_ / len(X_out) for k in range(1, 21)]
print("mean squared distance for 1 to 20 clusters:", np.round(msd, 2).tolist())
# L2 outcome clusters of every seed matched to L1 by overlap (Hungarian); the best of 20 is the variant used below
def match_to_L1(raw):
    ct = pd.crosstab(raw, L1_out).reindex(columns=range(1, 12), fill_value=0)
    r_, c_ = linear_sum_assignment(-ct.values)
    lab = np.zeros_like(L1_out)
    for a_, b_ in zip(r_, c_):
        lab[raw == ct.index[a_]] = b_ + 1
    return lab
L2_out_seeds = [match_to_L1(km_out[s_][2]) for s_ in seeds]
L2_out = L2_out_seeds[best_o]
print("agreement with L1 of each seed's matched clusters:", [round(float((l_ == L1_out).mean()), 3) for l_ in L2_out_seeds])
print("best of 20 (seed %d), agreement with L1: %.3f" % (best_o, (L2_out == L1_out).mean()))

**Result: partially reproduced.** Over 20 seeds the median largest cluster is 304.5 (322 +- 32.2) and
the median of the median sizes 161.0 (169 +- 16.9), both within tolerance, but the median smallest
cluster is 29.0, below 46 +- 4.6; single seeds give smallest clusters from 1 to 58. The mean squared
distance falls smoothly from 1 to 20 clusters.

## fig4c-positive-prediction-error

> PCA of post-shock firing of all 1812 neurons showed positive prediction error on top of shock
> responding as PC1 (17.7% of firing variance) ...

In [ ]:
coef_out, expl_out = bst3.pca(X_out)
p = coef_out[:, 0] * np.sign(coef_out[20:40, 0].mean())
w = dict(zip(["danger", "unc_shock", "unc_omission", "safety"], [p[i * 20:(i + 1) * 20].mean() for i in range(4)]))
print(f"PC1 {expl_out[0]:.2f}% (17.7 +- 1.77); PC2 {expl_out[1]:.2f}%")
print("mean PC1 weights:", {k: round(v, 4) for k, v in w.items()})
print("unc shock > danger:", w["unc_shock"] > w["danger"], "; danger > safety:", w["danger"] > w["safety"],
      "; omission not below safety:", w["unc_omission"] >= w["safety"])
C4 = bst3.sd_matrix(sheets, "Figure 4c", 2, 80)
print(f"Source Data Fig. 4c matrix {C4.shape}: PC1 {bst3.pca(C4)[1][0]:.2f}%; rows equal to a unit's outcome row:",
      sum(len(h) > 0 for h in bst3.match_rows(C4, X_out)[0]))
# the heat-plot version: baseline (mean over the four types of 500-ms bins 18:20, outcomeAnalysis.m) subtracted
base = np.mean([m[:, 17:20].mean(1) for m in bst3.type_means(z5, ["danger", "unc_shock", "unc_omission", "safety"])], 0)
print("rows equal to a unit's baseline-subtracted outcome row:", sum(len(h) > 0 for h in bst3.match_rows(C4, X_out - base[:, None])[0]))
print("\nPer animal (PC1 of that animal's units; mean weights oriented as above):")
for s_ in range(1, 11):
    c_, e_ = bst3.pca(X_out[subject == s_])
    q = c_[:, 0] * np.sign(c_[20:40, 0].mean())
    print(f"  subject {s_} ({int((subject == s_).sum())} units): PC1 {e_[0]:.1f}%, mean weights D, US, UO, S "
          f"{np.round([q[i * 20:(i + 1) * 20].mean() for i in range(4)], 3).tolist()}")

**Result: reproduced.** PC1 of the 1812 x 80 outcome matrix explains 17.68% (17.7 +- 1.77). Its mean
weights are uncertainty shock 0.194 > danger 0.071 > safety -0.032, and uncertainty omission (-0.005)
is not below safety. The Source Data Fig. 4c matrix (24.18%) holds the units' outcome rows with the
heat-plot baseline subtracted, as the counts above show, not the matrix PCA was run on. Per animal,
uncertainty shock has the largest mean PC1 weight in all 10 animals, and danger exceeds safety in 9
(not subject 7).

## fig4b-phasic-network and fig4b-tonic-network

> Outcome clusters k1 to k3 showed phasic firing changes after shock ... and by their correlations
> formed a phasic outcome network.

> Outcome clusters k6 to k9 showed sustained firing changes after surprising shock ... and by their
> correlations formed a tonic outcome network.

Cluster mean outcome profiles under L1 (and L2 as a variant). 500-ms intervals: 0-1 the first second
after shock offset, 0-9 the first 5 s, 8-9 the second ending 5 s after offset, 10-19 seconds 5 to 10.

In [ ]:
def outcome_profiles(lab):
    return np.array([X_out[lab == k].mean(0) for k in range(1, 12)])
def network_tests(lab, power=1):
    P = outcome_profiles(lab)
    D, US, UO, SF = P[:, :20], P[:, 20:40], P[:, 40:60], P[:, 60:80]
    shock = (D + US) / 2
    first = shock[:, 0:2].mean(1)
    ti_a = np.abs(first - shock[:, 0:10].mean(1))
    ti_b = np.abs(first - shock[:, 8:10].mean(1))
    R = np.corrcoef(P) ** power
    ph, to = [0, 1, 2], [5, 6, 7, 8]
    within_ph = R[np.ix_(ph, ph)][np.triu_indices(3, 1)].mean()
    between_ph = R[np.ix_(ph, [i for i in range(11) if i not in ph])].mean()
    within_to = R[np.ix_(to, to)][np.triu_indices(4, 1)].mean()
    between_to = R[np.ix_(to, [i for i in range(11) if i not in to])].mean()
    f1 = lambda M, k: M[k, 0:2].mean()
    res_ph = {"index t-a k1-k3 above k6-k9": bool(ti_a[ph].min() > ti_a[to].max()),
              "index t-b k1-k3 above k6-k9": bool(ti_b[ph].min() > ti_b[to].max()),
              "k1 US > D > 0 (1st s)": f1(US, 0) > f1(D, 0) > 0, "k2 D > US > 0": f1(D, 1) > f1(US, 1) > 0,
              "k3 D, US < 0": f1(D, 2) < 0 and f1(US, 2) < 0, "network": within_ph > between_ph}
    m = lambda M, k: M[k].mean()
    res_to = {"k6 US < 0 and lowest": m(US, 5) < 0 and m(US, 5) < min(m(D, 5), m(UO, 5), m(SF, 5)),
              "k7 US > 0 and highest": m(US, 6) > 0 and m(US, 6) > max(m(D, 6), m(UO, 6), m(SF, 6)),
              "k8 US > 0 and highest": m(US, 7) > 0 and m(US, 7) > max(m(D, 7), m(UO, 7), m(SF, 7)),
              "k7 sustained": US[6, 10:20].mean() > 0.5 * US[6, 0:2].mean(),
              "k8 sustained": US[7, 10:20].mean() > 0.5 * US[7, 0:2].mean(),
              "network": within_to > between_to}
    info = pd.DataFrame({"cluster": range(1, 12), "index t-a": ti_a.round(3), "index t-b": ti_b.round(3),
                         "D 1st s": D[:, 0:2].mean(1).round(2), "US 1st s": US[:, 0:2].mean(1).round(2),
                         "D mean": D.mean(1).round(2), "US mean": US.mean(1).round(2), "UO mean": UO.mean(1).round(2),
                         "S mean": SF.mean(1).round(2), "US 5-10 s": US[:, 10:20].mean(1).round(2)})
    return res_ph, res_to, info, (within_ph, between_ph, within_to, between_to)
for name, lab in (("L1", L1_out), ("L2", L2_out)):
    for power, pname in ((1, "r"), (2, "r squared")):
        ph, to, info, w = network_tests(lab, power)
        print(name, pname, "phasic:", ph)
        print(name, pname, "tonic:", to)
        print(name, pname, "mean within / between: phasic %.3f / %.3f, tonic %.3f / %.3f" % w)
    if name == "L1":
        print(info.to_string(index=False))
R = np.corrcoef(outcome_profiles(L1_out))
print("L1 cluster-cluster r (outcome profiles):")
print(pd.DataFrame(R, index=range(1, 12), columns=range(1, 12)).round(2).to_string())
# L2 for each of the 20 seeds (matched to L1): in how many seeds each part holds
held = Counter()
for lab in L2_out_seeds:
    for power, pname in ((1, "r"), (2, "r squared")):
        ph, to, _, _ = network_tests(lab, power)
        held.update({f"{pname}, phasic {k_}": int(bool(v_)) for k_, v_ in ph.items()})
        held.update({f"{pname}, tonic {k_}": int(bool(v_)) for k_, v_ in to.items()})
print("L2, seeds of 20 in which each part holds:")
for k_, v_ in held.items():
    print(f"    {k_}: {v_}")

**Result: fig4b-phasic-network and fig4b-tonic-network partially reproduced.** Under L1 every
profile condition holds: k1 to k3 have higher temporal indices than every one of k6 to k9 under both
t-a and t-b; in the first second k1 fires more to uncertainty shock than to danger (2.58, 1.59), k2
more to danger (2.85, 1.44), and k3 is suppressed on both (-0.97, -0.90); k6 is the only cluster
inhibited to surprising shock (-0.88, below its other means), and k7 and k8 increase most to it
(0.82, 0.31) with seconds 5 to 10 above half the first second. The network parts fail with the
Pearson correlation the criterion names: the mean r within k1 to k3 is -0.329 against 0.008 with the
other clusters, and within k6 to k9 -0.250 against 0.000, because k3 and k6 change firing in the
opposite direction to their fellow clusters. With r squared, the coefficient the Methods name
("Pearson's correlation coefficient (R2)") and the script plots, both network parts hold (0.539
against 0.076; 0.505 against 0.110), and both claims would be reproduced; whether the criterion
should allow r squared, as the one for fig2e does, is asked of a reader. Under L2 the tonic network
part holds in none of the 20 seeds with r and in all 20 with r squared, and the phasic network part
in 10 and 15.

## fig4e-network-pca

> Within the phasic outcome network, PC1 (37.0% of firing variance) was equivalent phasic shock firing
> on danger and uncertainty trials; within the tonic outcome network, PC1 (29.2%) was selective
> firing to surprising shock with opposing firing to safety.

In [ ]:
ph_units, to_units = np.isin(L1_out, [1, 2, 3]), np.isin(L1_out, [6, 7, 8, 9])
print("phasic units", int(ph_units.sum()), "; tonic units", int(to_units.sum()))
for name, M in (("recomputed", None), ("Source Data", "sd")):
    if M is None:
        Xp, Xt = X_out[ph_units], X_out[to_units]
    else:
        Xp, Xt = bst3.sd_matrix(sheets, "Figure 4e left", 2, 80), bst3.sd_matrix(sheets, "Figure 4e right", 2, 80)
    cp, ep = bst3.pca(Xp)
    ct_, et = bst3.pca(Xt)
    p = cp[:, 0] * np.sign(cp[0:2, 0].mean())
    D, US, UO, SF = p[:20], p[20:40], p[40:60], p[60:80]
    f = lambda v: v[0:2].mean()
    late = lambda v: v[10:20].mean()
    ph_ok = {"D 1st s > 2x late": f(D) > 2 * late(D), "US 1st s > 2x late": f(US) > 2 * late(US),
             "|D-US| < |D-S| and |US-S|": abs(f(D) - f(US)) < min(abs(f(D) - f(SF)), abs(f(US) - f(SF)))}
    q = ct_[:, 0] * np.sign(ct_[20:40, 0].mean())
    means = [q[i * 20:(i + 1) * 20].mean() for i in range(4)]
    to_ok = {"US largest": int(np.argmax(means)) == 1, "S negative": means[3] < 0}
    print(f"{name}: phasic PC1 {ep[0]:.2f}% (37.0 +- 3.7) {ph_ok}; first-second weights D {f(D):.3f} US {f(US):.3f} "
          f"UO {f(UO):.3f} S {f(SF):.3f}")
    print(f"{name}: tonic PC1 {et[0]:.2f}% (29.2 +- 2.92) {to_ok}; mean weights D, US, UO, S {np.round(means, 3).tolist()}")

**Result: reproduced.** The 265 phasic units give PC1 37.01% (37.0 +- 3.7) with danger and uncertainty
shock weights in the first second of 0.410 and 0.424, each more than twice their means over seconds 5
to 10 and close to each other; the 613 tonic units give PC1 29.20% (29.2 +- 2.92), largest for
uncertainty shock (0.202) and negative for safety (-0.078). The Source Data matrices give the same.

## fig4f-outcome-hubs

> In the phasic outcome network, k1 and k2 single-unit firing correlated better with the population
> firing of their fellow clusters than k3 did; in the tonic network, k6 ... was the hub.

Per unit, its 80-value row correlated with the mean row of each other cluster of its network,
squared and averaged (`outcomeAnalysis.m`); r without squaring is a variant. Two-sample t-tests
(MATLAB `ttest2`, equal variances), Bonferroni over the pairs within the network (3 phasic, 6 tonic).

In [ ]:
sd4f = pd.concat([pd.DataFrame([r for r in sheets[s][1:] if r[0] is not None], columns=["cluster", "R2"])
                  for s in ("Figure 4f left", "Figure 4f right")])
prof = outcome_profiles(L1_out)
def hub_values(net, sq):
    v = np.full(len(ids), np.nan)
    for u in np.flatnonzero(np.isin(L1_out, net)):
        k = L1_out[u]
        v[u] = bst3.unit_cluster_corr(X_out[u], [prof[j - 1] for j in net if j != k], sq)
    return v
for net, label in (([1, 2, 3], "phasic k1-k3"), ([6, 7, 8, 9], "tonic k6-k9"), ([1, 2, 3, 4, 5], "variant phasic k1-k5"),
                   ([7, 8, 9, 10], "variant tonic k7-k10")):
    for sq in (True, False):
        v = hub_values(net, sq)
        means = {k: round(np.nanmean(v[L1_out == k]), 3) for k in net}
        npairs = len(net) * (len(net) - 1) // 2
        tests = []
        for i, a_ in enumerate(net):
            for b_ in net[i + 1:]:
                t_ = stats.ttest_ind(v[L1_out == a_], v[L1_out == b_])
                lv = stats.levene(v[L1_out == a_], v[L1_out == b_], center="mean")
                tests.append(f"k{a_}-k{b_} t={t_.statistic:.2f} p={t_.pvalue:.2g}{'*' if t_.pvalue < 0.05 / npairs else ''} L={lv.pvalue:.2g}")
        same = ""
        if sq and label[0] != "v":
            mine = np.sort(v[np.isin(L1_out, net)])
            theirs = np.sort(sd4f[sd4f.cluster.isin(net)].R2.values)
            labels_sd = {k: int((sd4f.cluster == k).sum()) for k in net}
            same = (f"; all values equal Source Data as a set: {len(mine) == len(theirs) and np.allclose(mine, theirs, atol=1e-6)}; "
                    f"Source Data rows per label {labels_sd}, units per L1 cluster {dict((k, int((L1_out == k).sum())) for k in net)}")
        print(f"{label}, {'r squared' if sq else 'r'}: means {means}{same}")
        print("    ", "; ".join(tests))
# Source Data Fig. 4f right labels: outcomeAnalysis.m (line 936) labels the 613 values, held in the
# order k6, k7, k8, k9, as 142, 171, 111 and 189 rows, where k7 has 111 units and k8 171
v_t = hub_values([6, 7, 8, 9], True)
for lab_sd in (7, 8):
    vals = np.round(sd4f[sd4f.cluster == lab_sd].R2.values, 10)
    print(f"Source Data rows labelled {lab_sd}: {len(vals)}; values of k7 units {int(np.isin(vals, np.round(v_t[L1_out == 7], 10)).sum())}, "
          f"of k8 units {int(np.isin(vals, np.round(v_t[L1_out == 8], 10)).sum())}")
print("The tests above use the units' own clusters; k6's comparisons are with k7, k8 and k9 each, so the swap does not change them")

**Result: reproduced.** With the script's measure (r squared), whose per-unit values equal Source Data
Fig. 4f as a set, k1 (0.274) and k2 (0.260) each exceed k3 (0.148; p = 3.6e-12 and 9.8e-08, below
0.05/3), and k6 (0.264) has the highest tonic mean and exceeds k7, k8 and k9 (p <= 7.5e-07, below
0.05/6). With r the phasic part holds and k6 has the lowest tonic mean (-0.183). Source Data Fig. 4f
right labels 171 units as k7 and 111 as k8, where Figure 4b's sheets have 111 in k7 and 171 in k8.
The rows labelled k7 hold the 111 k7 values and 60 k8 values, as the script's hard-coded labels
give, so only the labels are swapped; the tests here use the units' own clusters.

## fig4g-outcome-anatomy

> Phasic outcome neurons were more common at the axis extremes ... while tonic outcome neurons, most
> strikingly the k6 hub, were distributed throughout the brainstem.

In [ ]:
ext = np.isin(region, [3, 4, 5, 7, 8, 9, 10, 20])     # DLPAG, LPAG, VLPAG, DRL, DRD, DRV, PDR, PMnR
fp, ft = ext[ph_units].mean(), ext[to_units].mean()
print(f"fraction in DLPAG, LPAG, VLPAG, DRL, DRD, DRV, PDR, PMnR: phasic {fp:.3f}, tonic {ft:.3f}, difference {fp - ft:.3f}")
print("regions spanned: phasic", len(set(region[ph_units])), "; tonic", len(set(region[to_units])))
print("regions per outcome cluster:", {f"k{k}": len(set(region[L1_out == k])) for k in range(1, 12)})
tab = pd.crosstab(ph_units[ph_units | to_units], ext[ph_units | to_units])
print("chi-square phasic/tonic x extremes (effect size only):", np.round(stats.chi2_contingency(tab)[:2], 3).tolist())

**Result: reproduced.** 0.600 of phasic units and 0.594 of tonic units are in DLPAG, LPAG, VLPAG, DRL,
DRD, DRV, PDR and PMnR (a difference of 0.006, chi-square p = 0.92), and the tonic network spans 20
regions against 17. K6 spans 17 regions, against 10 to 19 for the other outcome clusters. The effect
is small, as on the paper's own values.

## fig5a-outcome-cluster-signals

> Regression revealed distinct shock and prediction error signals across the 11 outcome clusters;
> k1 neurons transiently signaled both sensory shock and prediction error, while k6 neurons
> signaled only error.

The script's beta matrix holds prediction error in columns 1-20 and shock in 21-40
(`outcomeRegression.m`: `regTerms = [blank, regPos, regShock]`, b(2) to columns 1-20); the claims
file reads columns 1-20 as shock. Which order Source Data Fig. 5b holds is checked below.

In [ ]:
sd5b = bst3.sd_matrix(sheets, "Figure 5b middle", 2, 40)
obetas = {}
for mode in ("r-a", "r-b"):
    for ic, iname in ((True, "i-a"), (False, "i-b")):
        obetas[(mode, iname)] = bst3.outcome_betas(z5, L1_out, range(1, 12), mode, ic)
for key, B in obetas.items():
    print(key, "max abs diff from Source Data Fig. 5b middle: as is %.2e; halves swapped %.2e" %
          (np.abs(B - sd5b).max(), np.abs(np.hstack([B[:, 20:], B[:, :20]]) - sd5b).max()))
def fig5a_tests(B):
    pe, sh = B[:, :20], B[:, 20:]
    k1 = {}
    for name, v in (("shock", sh[0]), ("PE", pe[0])):
        a_ = np.abs(v)
        k1[name] = int(np.argmax(a_)) < 4 and a_[10:20].max() < 0.5 * a_.max()
    k6 = np.abs(pe[5]).mean() >= 2 * np.abs(sh[5]).mean()
    return {"k1 transient shock": k1["shock"], "k1 transient PE": k1["PE"], "k6 PE only": bool(k6),
            "k6 mean |PE| / mean |shock|": round(np.abs(pe[5]).mean() / np.abs(sh[5]).mean(), 2)}
for key, B in obetas.items():
    print(key, fig5a_tests(B))
B = obetas[("r-a", "i-a")]
print("\nk1 shock betas:", B[0, 20:].round(2).tolist())
print("k1 PE betas:   ", B[0, :20].round(2).tolist())
print("k6 shock betas:", B[5, 20:].round(2).tolist())
print("k6 PE betas:   ", B[5, :20].round(2).tolist())

**Result: reproduced.** Under all four readings k1's shock and prediction error betas peak in the
first 2 s and stay below half that peak from 5 to 10 s, and k6's mean absolute prediction error beta is
12.3 times its mean absolute shock beta (8.9 without an intercept). The script's betas equal Source
Data Fig. 5b middle with prediction error in columns 1 to 20, the reverse of the Methods' layout.

## fig5b-outcome-signal-pca

> PCA of outcome cluster beta weights showed opposing sensory shock and prediction error signals as
> PC1 (53.7% of signaling variance) and dynamic sensory shock to prediction error signaling as PC2
> (33.4%).

In [ ]:
def out_shape(c, intact2=None):
    p1, p2 = c[:, 0], c[:, 1]
    pe1, sh1, pe2, sh2 = p1[:20], p1[20:], p2[:20], p2[20:]
    res = {"PC1 opposing": np.sign(sh1.mean()) != np.sign(pe1.mean()),
           "PC2 |shock| peak in first 2 s": int(np.argmax(np.abs(sh2))) < 4,
           "PC2 |PE| peak after 2 s": int(np.argmax(np.abs(pe2))) >= 4}
    return res
B_out = obetas[("r-a", "i-a")]
pcs_out = {}
for name, M in (("Source Data Fig. 5b middle", sd5b), ("recomputed L1 (r-a, i-a)", B_out),
                ("recomputed L2 (r-a, i-a)", bst3.outcome_betas(z5, L2_out, range(1, 12), "r-a", True))):
    c, e = bst3.pca(M)
    pcs_out[name] = (c, e)
    print(f"{name}: PC1 {e[0]:.2f}% (53.7 +- 5.37), PC2 {e[1]:.2f}% (33.4 +- 3.34); {out_shape(c)}")
c, e = pcs_out["recomputed L1 (r-a, i-a)"]
print("PC1 PE weights:", c[:20, 0].round(3).tolist())
print("PC1 shock weights:", c[20:, 0].round(3).tolist())
print("PC2 PE weights:", c[:20, 1].round(3).tolist())
print("PC2 shock weights:", c[20:, 1].round(3).tolist())

**Result: reproduced.** The recomputed 11 x 40 betas (equal to Source Data Fig. 5b middle) give PC1
53.74% and PC2 33.43%, within tolerance; PC1's shock and prediction error weights have opposite
signs, and PC2's shock weights peak in the first 2 s and its prediction error weights later. L2
gives 47.80 and 30.92%, out of tolerance, with the same shapes.

## fig5c-phasic-lesion and fig5d-tonic-lesion

> Lesioning the phasic outcome network left signaling dominated by sustained prediction error (80.0%
> of signaling variance), with residual sustained shock (16.8%) ...

> Lesioning the tonic outcome network left opposing, phasic sensory shock and prediction error
> signaling as PC1 (61.3% of signaling variance) and dynamic sensory shock to prediction error as PC2
> (29.8%), similar to that of all neurons.

Lesion as in `outcomeRegression.m` (s-d): the shuffled clusters' firing permuted across the 16
trial rows before regression, betas averaged over 1000 iterations, one PCA (v-a). Whether one row
permutation is used for all 20 intervals (s-d1) or one per interval (s-d2) is not settled by the
script; both are run. s-a and s-b (permuting betas) are variants.

In [ ]:
rng = np.random.default_rng(7)
olesion = {}
for name, shuffled in (("phasic", {1, 2, 3}), ("tonic", {6, 7, 8, 9})):
    olesion[(name, "s-d1")] = bst3.outcome_lesion_betas(z5, L1_out, range(1, 12), shuffled, rng, 1000, True)
    olesion[(name, "s-d2")] = bst3.outcome_lesion_betas(z5, L1_out, range(1, 12), shuffled, rng, 1000, False)
    rws = np.array(sorted(shuffled)) - 1
    olesion[(name, "s-a")] = bst3.beta_shuffle_average(B_out, rws, rng, 1000, "s-a")
    olesion[(name, "s-b")] = bst3.beta_shuffle_average(B_out, rws, rng, 1000, "s-b")
olesion[("phasic", "Source Data")] = bst3.sd_matrix(sheets, "Figure 5c middle", 2, 40)
olesion[("tonic", "Source Data")] = bst3.sd_matrix(sheets, "Figure 5d middle", 2, 40)
ic, ie_ = pcs_out["recomputed L1 (r-a, i-a)"]
rows = []
for (name, reading), M in olesion.items():
    c, e = bst3.pca(M)
    pe1, sh1, pe2, sh2 = c[:20, 0], c[20:, 0], c[:20, 1], c[20:, 1]
    if name == "phasic":
        ok = {"PC1 |PE| > |shock|": np.abs(pe1).mean() > np.abs(sh1).mean(),
              "PC1 PE sustained (1-10 s >= half max)": bool((np.abs(pe1[2:]) >= 0.5 * np.abs(pe1).max()).all()),
              "PC2 |shock| > |PE|": np.abs(sh2).mean() > np.abs(pe2).mean()}
        exp = "(80.0 +- 8.0, 16.8 +- 1.68)"
    else:
        v2 = c[:, 1] * (1 if c[:, 1] @ ic[:, 1] >= 0 else -1)
        ok = {"PC1 opposing": np.sign(sh1.mean()) != np.sign(pe1.mean()),
              "PC1 both peak in first 2 s": int(np.argmax(np.abs(sh1))) < 4 and int(np.argmax(np.abs(pe1))) < 4,
              **{k: v for k, v in out_shape(c).items() if k.startswith("PC2")},
              "r with intact PC2 > 0.5": round(float(np.corrcoef(v2, ic[:, 1])[0, 1]), 3)}
        exp = "(61.3 +- 6.13, 29.8 +- 2.98)"
    rows.append({"lesion": name, "reading": reading, "PC1 %": round(e[0], 2), "PC2 %": round(e[1], 2), "expected": exp, **ok})
print(pd.DataFrame(rows).to_string(index=False))
print("max abs diff from Source Data, phasic s-d1 %.3f, s-d2 %.3f; tonic s-d1 %.3f, s-d2 %.3f" % (
    np.abs(olesion[("phasic", "s-d1")] - olesion[("phasic", "Source Data")]).max(),
    np.abs(olesion[("phasic", "s-d2")] - olesion[("phasic", "Source Data")]).max(),
    np.abs(olesion[("tonic", "s-d1")] - olesion[("tonic", "Source Data")]).max(),
    np.abs(olesion[("tonic", "s-d2")] - olesion[("tonic", "Source Data")]).max()))

**Result: fig5c-phasic-lesion and fig5d-tonic-lesion reproduced.** With the script's lesion (s-d,
one permutation for all intervals or one per interval) the phasic lesion gives 79.99 and 16.81%
(80.0 and 16.8 reported), PC1 a sustained prediction error signal and PC2 mostly shock; the tonic
lesion gives 61.18 and 29.88% (61.3 and 29.8), PC1 opposing shock and prediction error with both
peaks in the first 2 s, and PC2 dynamic and correlated with the intact PC2 at r = 0.895. Shuffling
betas (s-a, s-b) gives 76.50 to 78.59% and 53.45 to 60.93% for PC1, with the same shapes.

## fig5e-no-anatomical-bias

> Anatomical biases for phasic and tonic outcome network neurons were absent.

Per region, its share of tonic units minus its share of phasic units (percentage points); phasic as
k1-k3 (p-a) or k1-k5 (p-b). Judged against 4.48 and 1.5 x 4.48 = 6.72.

In [ ]:
sd5e = np.array(sheets["Figure 5e right"][1][1:22], float)
for name, ph in (("p-a k1-k3", [1, 2, 3]), ("p-b k1-k5", [1, 2, 3, 4, 5])):
    d5e, a, b = bst3.differential(region, np.isin(L1_out, [6, 7, 8, 9]), np.isin(L1_out, ph))
    print(f"{name}: largest |differential| {np.abs(d5e).max():.2f} ({bst3.REGIONS[int(np.argmax(np.abs(d5e)))]})")
    print("   ", dict(zip(bst3.REGIONS, d5e.round(2))))
    tab = np.vstack([a, b])
    tab = tab[:, tab.sum(0) > 0]
    chi = stats.chi2_contingency(tab)
    print(f"    chi-square region x network (variant): chi2 = {chi[0]:.1f}, df {chi[2]}, p = {chi[1]:.2g}")
print("Source Data Fig. 5e right:", dict(zip(bst3.REGIONS, sd5e.round(2))), "; max |.|", np.abs(sd5e).max().round(2))
# A reference for the largest differential: the network labels permuted among the units of the two
# networks, 1000 times (a variant, not the judge)
rng = np.random.default_rng(8)
for name, ph in (("p-a k1-k3", [1, 2, 3]), ("p-b k1-k5", [1, 2, 3, 4, 5])):
    to_m, ph_m = np.isin(L1_out, [6, 7, 8, 9]), np.isin(L1_out, ph)
    pool, n_to = np.flatnonzero(to_m | ph_m), int(to_m.sum())
    obs = np.abs(bst3.differential(region, to_m, ph_m)[0]).max()
    null = np.empty(1000)
    for i in range(1000):
        perm = rng.permutation(pool)
        a_, b_ = np.zeros(len(region), bool), np.zeros(len(region), bool)
        a_[perm[:n_to]], b_[perm[n_to:]] = True, True
        null[i] = np.abs(bst3.differential(region, a_, b_)[0]).max()
    print(f"{name}: largest |differential| {obs:.2f}; with permuted labels median {np.median(null):.2f}, "
          f"95th percentile {np.percentile(null, 95):.2f}, fraction at least as large {(null >= obs).mean():.3f}")

**Result: reproduced, resting on an assumption.** With phasic read as k1 to k5 (p-b), the reading
behind the paper's Fig. 5e (its Source Data), the largest differential is 4.48 points (MnR) and every
value equals Source Data, at the limit the criterion sets. This reading cannot fail once the data
match Source Data, since the criterion's threshold is the largest Source Data value of the same
quantity; the threshold is asked of a reader. With phasic as k1 to k3 (p-a), the phasic network the
Results define, the largest differential is 9.82 (PMnR), above 1.5 x 4.48, which would not
reproduce the claim. With network labels permuted among the units of the two networks, the 95th
percentile of the largest differential is 5.88 (p-a) and 4.64 (p-b); 9.82 is reached in 0.001 of
the permutations and 4.48 in 0.058. The chi-square of region by network, shown as a variant, gives
p = 0.039 (p-b) and 3e-05 (p-a). The outcome follows p-b, the paper's own computation for the
figure, since a claim of absence is judged on the paper's test as written.

## cue-outcome-membership-independent

> Membership in cue networks did not influence membership in outcome networks; neither the phasic
> (85/265, 32.1%) nor the tonic (159/613, 25.9%) outcome neurons differed from the 27.9% expected in
> the cue subnetwork.

In [ ]:
for lname, lc, lo in (("L1", L1_cue, L1_out), ("L2", L2_cue, L2_out)):
    sub = np.isin(lc, range(1, 10))
    for net, members in (("phasic", [1, 2, 3]), ("tonic", [6, 7, 8, 9])):
        m = np.isin(lo, members)
        k, n = int((sub & m).sum()), int(m.sum())
        chi = stats.chisquare([k, n - k], [n * 0.279, n * 0.721])
        chi2 = stats.chi2_contingency(pd.crosstab(m, sub), correction=False)
        print(f"{lname} {net}: {k}/{n} = {k / n:.3f} in the cue subnetwork; goodness of fit vs 0.279 chi2 = {chi.statistic:.2f}, "
              f"p = {chi.pvalue:.2f}; 2 x 2 chi2 = {chi2[0]:.2f}, p = {chi2[1]:.2f}")
# L2 for each of the 20 seeds: the cue solution of seed s (numbered by contribution) with the
# outcome solution of seed s (matched to L1)
rows = []
for s_ in seeds:
    sub = np.isin(L2_cue_seeds[s_], range(1, 10))
    row = {"seed": s_}
    for net, mem in (("phasic", [1, 2, 3]), ("tonic", [6, 7, 8, 9])):
        m = np.isin(L2_out_seeds[s_], mem)
        k, n = int((sub & m).sum()), int(m.sum())
        row[f"{net} in subnetwork"] = f"{k}/{n}"
        row[f"{net} fit p"] = stats.chisquare([k, n - k], [n * 0.279, n * 0.721]).pvalue
        row[f"{net} 2x2 p"] = stats.chi2_contingency(pd.crosstab(m, sub), correction=False)[1]
    rows.append(row)
l2m = pd.DataFrame(rows)
print(l2m.to_string(index=False, float_format=lambda x: f"{x:.2g}"))
print("median p over the 20 seeds:", l2m[[c for c in l2m if c.endswith(" p")]].median().round(3).to_dict())
print("seeds in which neither goodness-of-fit test has p < 0.05:",
      int(((l2m["phasic fit p"] >= 0.05) & (l2m["tonic fit p"] >= 0.05)).sum()), "of 20; neither 2 x 2 test:",
      int(((l2m["phasic 2x2 p"] >= 0.05) & (l2m["tonic 2x2 p"] >= 0.05)).sum()), "of 20")

**Result: reproduced.** Under L1, 85 of 265 phasic units (0.321) and 159 of 613 tonic units (0.259)
are in the cue subnetwork; against 0.279 the goodness-of-fit chi-square is 2.30 (p = 0.13) and 1.17
(p = 0.28), and the 2 x 2 tests give p = 0.10 and 0.19. Under L2 the result depends on the
clustering: over the 20 seeds the median goodness-of-fit p is 0.048 (phasic) and 0.011 (tonic), and
neither goodness-of-fit test has p < 0.05 in 4 of the 20 seeds. The outcome is judged on the
paper's own membership (L1), the networks whose counts the claim gives.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| source-data-cluster-link | reproduced |
| fig1c-suppression-order | reproduced |
| figs2-each-rat-discriminates | reproduced |
| units-sessions-subjects | reproduced |
| fig1e-regions | reproduced |
| fig2b-cue-clusters | reproduced |
| fig2c-cue-pc1-ordered | reproduced |
| fig2c-pc1-contribution | reproduced |
| fig2d-danger-latency | not reproduced |
| fig2e-cue-subnetwork | reproduced |
| fig2f-cue-hubs | partially reproduced (rests on an assumption) |
| fig2g-cue-anatomy | reproduced |
| fig3a-cluster-signals | partially reproduced |
| fig3b-signal-pca | reproduced |
| fig3c-subnetwork-lesion | partially reproduced |
| fig3d-supranetwork-lesion | reproduced |
| fig3e-network-anatomy | reproduced |
| fig4b-outcome-clusters | partially reproduced |
| fig4c-positive-prediction-error | reproduced |
| fig4b-phasic-network | partially reproduced |
| fig4b-tonic-network | partially reproduced |
| fig4e-network-pca | reproduced |
| fig4f-outcome-hubs | reproduced |
| fig4g-outcome-anatomy | reproduced |
| fig5a-outcome-cluster-signals | reproduced |
| fig5b-outcome-signal-pca | reproduced |
| fig5c-phasic-lesion | reproduced |
| fig5d-tonic-lesion | reproduced |
| fig5e-no-anatomical-bias | reproduced (rests on an assumption) |
| cue-outcome-membership-independent | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** It runs in Python on the cube file and the spike
times, following the paper's MATLAB scripts where the Methods leave a detail open: the
normalization of `cubeScript.m`, the 1-s and 500-ms bins of the scripts' matrices, the regression
with an intercept on cluster mean firing, the shuffles of the contribution and lesion analyses. The
paper's k-means clusters cannot be regenerated (the script seeds MATLAB's generator), so the
paper's own membership is taken from its Source Data, which every unit matches exactly; this
attempt's own k-means is shown beside it. The cue hub measure uses r squared, the measure the
Methods name; with r that outcome would be lower. The outcome networks are judged with r, as their
criteria say; with r squared both would be reproduced. The danger latency is searched within the
10-s cue as the Methods say; no reading tried reproduces the Source Data latencies, and no script
computes them. This attempt's own k-means (L2) is reported as the median over 20 seeds where a
claim's test turns on it (fig2c-pc1-contribution, cue-outcome-membership-independent); the best of
the 20 is a variant elsewhere.

**What was found about the data set.** The cube file (its 1-s and 500-ms z-scored firing)
equals the spike times binned and normalized as `cubeScript.m` does, the suppression ratios equal those recomputed from the poke times in all 73
sessions, and the unit, female and per-subject counts equal the paper's. 204 unit files hold
repeated spike times (9289 in all), which the script's counting drops. The data set has 73
sessions where Supplementary Table 1 lists 75 (one fewer for subjects 5 and 10). One session
(js27101920) has 5 shocks, with none on its first danger trial. In session js10030220 three unit
files carry event times that differ from the others. Region names are stored as MATLAB string
objects that cannot be read outside MATLAB, and the region numbers need the scripts' mapping,
including their correction of 35 units numbered as CIC to DpWh. The data set holds no cluster
membership, so the paper's clusters can only be recovered from its Source Data.